## 1. Environment Setup and Memory Optimization

Configure PyTorch CUDA memory allocator to use expandable segments. This prevents Out-of-Memory (OOM) errors by allowing more flexible GPU memory allocation during training.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  

# Notebook Configuration Summary

## Training Approach

This notebook uses **4-bit quantization (QLoRA)** for memory-efficient training:

### Configuration:

1. **Dense Model**: Loaded with 4-bit NF4 quantization via BitsAndBytesConfig
2. **MoE Model**: 8 experts per layer with top-2 routing, 4-bit quantized experts
3. **Training**: QLoRA with paged_adamw_8bit optimizer

### Memory Requirements:

- Dense model: ~4-5GB GPU memory (4-bit quantized)
- MoE model: ~25-30GB GPU memory (8 experts, 4-bit)
- Training: ~35-45GB GPU memory (with gradient checkpointing)

**Note**: Requires GPU with at least 48GB memory (e.g., A6000, A100, H100) for training

---


## 2. Install Required Libraries

Install all dependencies needed for:
- Dataset handling (kagglehub, datasets)
- Model loading (transformers, bitsandbytes, accelerate)
- Evaluation (evaluate, scikit-learn)
- Experiment tracking (wandb, tensorboard)
- Utilities (torch, tqdm)

In [2]:
import sys
print(f"Python executable: {sys.executable}")

!{sys.executable} -m pip install -q kagglehub evaluate wandb tensorboard bitsandbytes accelerate peft transformers torch scikit-learn tqdm matplotlib seaborn

import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries installed successfully!")

Python executable: /root/shit/.venv/bin/python
All libraries installed successfully!


In [3]:
# Additional imports needed throughout the notebook
from collections import defaultdict
import psutil
import warnings
warnings.filterwarnings('ignore')


## 3. Download MMLU Dataset

Download the MMLU dataset from Kaggle. The dataset contains:
- **train.csv**: Training questions with answers
- **test.csv**: Test questions
- **valid.csv**: Validation questions

Each row has: question prompt, 4 answer options (A-D), and the correct answer.

In [4]:
import kagglehub

# Download the MMLU dataset
path = kagglehub.dataset_download("peiyuanliu2001/mmlu-dataset")

print(f"Dataset downloaded to: {path}")
print(f"Available files: {os.listdir(path)}")

Dataset downloaded to: /root/.cache/kagglehub/datasets/peiyuanliu2001/mmlu-dataset/versions/2
Available files: ['test.csv', 'train.csv', 'valid.csv']


## 4. Load and Explore Dataset

Load the training data into a pandas DataFrame and examine its structure, distribution, and sample questions.

In [5]:
import pandas as pd
from datasets import Dataset

# Load the dataset
df = pd.read_csv(f"{path}/train.csv")

print("Dataset Overview:")
print(f"  Total samples: {len(df):,}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\nAnswer distribution:")
print(df['answer'].value_counts().to_dict())

print("\nSample questions:")
display(df.head())

Dataset Overview:
  Total samples: 98,487
  Columns: ['prompt', 'A', 'B', 'C', 'D', 'answer']

Answer distribution:
{'C': 26491, 'B': 25377, 'D': 24767, 'A': 21852}

Sample questions:


,prompt,A,B,C,D,answer
0,Davis decided to kill Adams. He set out for Ad...,Adams only.,Brooks only.,Case only.,Adams and Brooks,B
1,A state statute requires any person licensed t...,"guilty, because this is a public welfare offen...","guilty, because he cannot be excused on the ba...","not guilty, because the statute punishes omiss...","not guilty, because he was not aware of the va...",D
2,"Lender met Borrower on the street, demanded th...","Yes, because Mann threatened to use deadly for...","Yes, unless Mann was related to Borrower.","No, if it was apparent that Lender was about t...","No, because Lender was the original aggressor ...",C
3,Peter sued Don for breach of contract. The cou...,must permit Don to answer if he had objected t...,"may permit Don to answer, whether or not he ha...",may permit Don to answer only if he had object...,"cannot permit Don to answer, whether or not he...",B
4,Ames had painted Bell's house under a contract...,partial breach of contract only if Ames had pr...,partial breach of contract whether or not Ames...,total breach of contract only if Ames had prop...,total breach of contract whether or not Ames h...,C


## Description

* Split data and format prompts for 0-shot evaluation.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [6]:
from sklearn.model_selection import train_test_split

def format_mmlu_prompt(row):
    """Format MMLU questions for evaluation."""
    prompt = f"""Question: {row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}

Answer:"""
    return prompt

stratify_labels = df['subject'] if 'subject' in df.columns else None
train_indices, eval_indices = train_test_split(
    df.index,
    test_size=0.3,
    random_state=42,
    stratify=stratify_labels if stratify_labels is not None else None,
)

df['split'] = 'train'
df.loc[eval_indices, 'split'] = 'eval'
train_subset_for_examples = df.loc[train_indices].copy()


print("\nUsing 0-SHOT prompting\n")
df['formatted_prompt'] = df.apply(format_mmlu_prompt, axis=1)

answer_to_idx = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['answer_idx'] = df['answer'].map(answer_to_idx)

print("\nSample formatted prompt:")

sample_prompt = df['formatted_prompt'].iloc[0]
print(sample_prompt[:500] + "..." if len(sample_prompt) > 500 else sample_prompt)

print(f"\nExpected answer: {df['answer'].iloc[0]}")
print(f"rompt length: {len(sample_prompt)} characters")

train_df = df[df['split'] == 'train'].copy()
eval_df = df[df['split'] == 'eval'].copy()

print(f"\nSplit summary:")
print(f"  Train samples: {len(train_df):,}")
print(f"  Eval samples:  {len(eval_df):,}")



Using 0-SHOT prompting


Sample formatted prompt:
Question: Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the intended victim(s) was/were

A. Adams only.
B. Brooks only.
C. Case only.
D. Adams and Brooks

Answer...

Expected answer: B
rompt length: 501 characters

Split summary:
  Train samples: 68,940
  Eval samples:  29,547


## 7. Prepare Evaluation Dataset

Prepare the dataset for evaluation:
- Use a subset of the data for evaluation
- Convert to HuggingFace Dataset format for easy processing

In [7]:
from datasets import Dataset, DatasetDict

# Determine columns to include
columns_to_use = ['prompt', 'formatted_prompt', 'answer', 'answer_idx']
if 'subject' in train_df.columns:
    columns_to_use.insert(0, 'subject')
if 'A' in train_df.columns:
    columns_to_use.extend(['A', 'B', 'C', 'D'])

train_dataset_raw = Dataset.from_pandas(
    train_df[columns_to_use],
    preserve_index=False,
)
eval_dataset_raw = Dataset.from_pandas(
    eval_df[columns_to_use],
    preserve_index=False,
)

dataset = DatasetDict({
    'train': train_dataset_raw,
    'test': eval_dataset_raw,
})
eval_dataset = eval_dataset_raw

print("Dataset prepared:")
print(f"  Training samples: {len(train_dataset_raw):,}")
print(f"  Evaluation samples: {len(eval_dataset_raw):,}")
print(f"  Columns: {eval_dataset_raw.column_names}")


Dataset prepared:
  Training samples: 68,940
  Evaluation samples: 29,547
  Columns: ['prompt', 'formatted_prompt', 'answer', 'answer_idx', 'A', 'B', 'C', 'D']


## 8. Load Model with 4-bit Quantization (QLoRA)

Load Mistral-7B-v0.1 with 4-bit quantization:
- **NormalFloat4 (NF4)**: Optimal quantization format for normally distributed weights
- **Double quantization**: Further compress quantization constants  
- **BFloat16 compute**: Use bfloat16 for computations despite 4-bit storage

**Memory Savings**: ~14GB → ~4GB (3.5x reduction), enabling training on consumer GPUs.

In [8]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define the model path
model_id = "mistralai/Mistral-7B-v0.1"

# Updated quantization config for 4-bit with CPU offload support
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
    llm_int8_enable_fp32_cpu_offload=True  # Safe to include for 4-bit
)

# Custom device map: everything on GPU, or specify modules to offload if needed
# For most users, this will keep all on GPU. If you want to offload, e.g., 'lm_head': 'cpu'
device_map = {"": "cuda:0"}

print("Loading model with 4-bit quantization (QLoRA)...")
print("This will require ~4-5GB GPU memory (vs ~14GB full precision)")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True
)

# Store reference for baseline evaluation
base_model = model

Loading model with 4-bit quantization (QLoRA)...
This will require ~4-5GB GPU memory (vs ~14GB full precision)


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.14s/it]


## Description

* Load the tokenizer.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

## 9. Get Answer Token IDs

Extract token IDs for 'A', 'B', 'C', 'D' from the tokenizer. These are needed for MMLU evaluation where we compare the model's probability distribution over these specific tokens (rather than generating full text responses).

## Description

* Extract token IDs for answer choices (A, B, C, D) used in MMLU evaluation.
* This cell implements the functionality described above
* Results and outputs are displayed below


## Description

* Map answer choices to token IDs.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [10]:
# Get token IDs for answer choices
ANSWER_TOKENS = {
    'A': tokenizer.encode('A', add_special_tokens=False)[0],
    'B': tokenizer.encode('B', add_special_tokens=False)[0],
    'C': tokenizer.encode('C', add_special_tokens=False)[0],
    'D': tokenizer.encode('D', add_special_tokens=False)[0],
}

print("Answer token IDs:")
for letter, token_id in ANSWER_TOKENS.items():
    decoded = tokenizer.decode([token_id])
    print(f"  {letter}: {token_id} → '{decoded}'")

# Also create reverse mapping
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

Answer token IDs:
  A: 330 → 'A'
  B: 365 → 'B'
  C: 334 → 'C'
  D: 384 → 'D'


## 10. Set Maximum Sequence Length

Set the maximum sequence length for tokenization. Set appropriate length for 0-shot evaluation.

In [11]:
# Maximum sequence length for 0-shot prompting

MAX_LENGTH = 512  # 0-shot prompting
print(f"Maximum sequence length set to: {MAX_LENGTH} (0-shot mode)")


# Verify Mistral-7B can handle this length
print(f"\n Context window:")
print(f" Model max: 8192 tokens (Mistral-7B sliding window)")
print(f" Our limit: {MAX_LENGTH} tokens")
print(f" Safety margin: {8192 - MAX_LENGTH} tokens")

Maximum sequence length set to: 512 (0-shot mode)

 Context window:
 Model max: 8192 tokens (Mistral-7B sliding window)
 Our limit: 512 tokens
 Safety margin: 7680 tokens


## 11. Define Evaluation Functions

- MMLU Accuracy (overall + per-category)
- Training/Eval Loss
- Perplexity
- Throughput & Latency
- GPU Memory
- Confidence Calibration (ECE)
- Top-2 Accuracy
- Macro-averaged Accuracy
- Confusion Matrix

In [12]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import time

def compute_ece(confidences, predictions, labels, n_bins=10):
    """
    Compute Expected Calibration Error (ECE).
    Measures how well model confidence aligns with actual accuracy.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()

        if prop_in_bin > 0:
            accuracy_in_bin = (predictions[in_bin] == labels[in_bin]).mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece

def evaluate_mmlu_comprehensive(model, tokenizer, eval_dataset, answer_tokens,
                                 device="cuda", max_samples=None, show_progress=True):
    """
    Comprehensive MMLU evaluation with all Priority 1-3 metrics.

    Returns dict with:
    - accuracy: Overall accuracy
    - top2_accuracy: Top-2 accuracy
    - confidences: Model confidence scores
    - predictions: Predicted answers
    - true_labels: Ground truth answers
    - confusion_matrix: Confusion matrix
    - ece: Expected Calibration Error
    - throughput: Samples per second
    - avg_latency: Average latency per sample
    """
    model.eval()

    correct = 0
    top2_correct = 0
    total = 0

    all_confidences = []
    all_predictions = []
    all_true_labels = []

    # Limit samples if specified
    samples = eval_dataset
    if max_samples is not None:
        samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    # Get token IDs as tensor
    answer_token_ids = torch.tensor([answer_tokens['A'], answer_tokens['B'],
                                      answer_tokens['C'], answer_tokens['D']])

    # Start timing
    start_time = time.time()

    iterator = tqdm(samples, desc="Evaluating", disable=not show_progress)

    with torch.no_grad():
        for example in iterator:
            prompt = example['formatted_prompt']
            true_answer = example['answer']
            true_idx = answer_to_idx[true_answer]

            # Tokenize and get logits
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, :]

            # Get probabilities for A, B, C, D
            answer_logits = last_logits[answer_token_ids]
            answer_probs = torch.softmax(answer_logits, dim=0)

            # Top-1 prediction
            pred_idx = answer_probs.argmax().item()
            pred_answer = idx_to_letter[pred_idx]
            confidence = answer_probs[pred_idx].item()

            # Top-2 prediction
            top2_indices = answer_probs.topk(2).indices.tolist()

            # Store results
            all_confidences.append(confidence)
            all_predictions.append(pred_idx)
            all_true_labels.append(true_idx)

            if pred_answer == true_answer:
                correct += 1
            if true_idx in top2_indices:
                top2_correct += 1
            total += 1

    # End timing
    end_time = time.time()
    duration = end_time - start_time

    # Convert to numpy arrays
    confidences = np.array(all_confidences)
    predictions = np.array(all_predictions)
    true_labels = np.array(all_true_labels)

    # Compute metrics
    accuracy = correct / total if total > 0 else 0.0
    top2_accuracy = top2_correct / total if total > 0 else 0.0
    ece = compute_ece(confidences, predictions, true_labels)
    conf_matrix = confusion_matrix(true_labels, predictions, labels=[0, 1, 2, 3])

    # Performance metrics
    throughput = total / duration if duration > 0 else 0.0
    avg_latency = duration / total if total > 0 else 0.0

    return {
        'accuracy': accuracy,
        'top2_accuracy': top2_accuracy,
        'correct': correct,
        'total': total,
        'ece': ece,
        'confidences': confidences,
        'predictions': predictions,
        'true_labels': true_labels,
        'confusion_matrix': conf_matrix,
        'throughput': throughput,
        'avg_latency': avg_latency,
    }

print("evaluation functions defined")

evaluation functions defined


## 11.1 Enhanced Evaluation Metrics

Additional comprehensive metrics for MoE analysis:
- **Computational Efficiency**: FLOPs, tokens/sec, ms/token
- **Parameter Efficiency**: Active params, trainable params, sparsity ratio
- **Scalability Metrics**: Memory usage, throughput scaling

In [13]:
import time
import gc

def compute_model_flops(model, seq_length=512):
    """
    Estimate FLOPs per forward pass for transformer model.

    Formula: FLOPs ≈ 2 * active_params * seq_length
    For MoE: multiply by sparsity factor (active_experts / total_experts)
    """
    config = model.config
    n_layers = config.num_hidden_layers
    d_model = config.hidden_size
    intermediate_size = config.intermediate_size
    vocab_size = config.vocab_size

    # Attention FLOPs per layer: 4 * seq_len * d_model^2
    attention_flops = 4 * seq_length * d_model * d_model * n_layers

    # FFN FLOPs per layer: 2 * seq_len * d_model * intermediate_size * 2
    ffn_flops = 8 * seq_length * d_model * intermediate_size * n_layers

    total_flops = attention_flops + ffn_flops

    # Check if MoE model
    is_moe = False
    sparsity_factor = 1.0
    
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                num_experts_per_tok = layer.mlp.num_experts_per_tok
                sparsity_factor = num_experts_per_tok / num_experts
                break
    except:
        pass

    if is_moe:
        # Apply sparsity to FFN only (attention is unchanged)
        total_flops = attention_flops + (ffn_flops * sparsity_factor)

    return total_flops


def compute_throughput_metrics(model, tokenizer, eval_dataset, max_samples=100):
    """
    Measure throughput and latency metrics.

    Returns:
        - tokens_per_second: Throughput in tokens/sec
        - ms_per_token: Latency in ms/token
        - samples_per_second: Sample throughput
        - total_time: Total evaluation time
    """
    model.eval()

    # Take subset
    samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    total_tokens = 0
    sample_times = []

    with torch.no_grad():
        for example in samples:
            prompt = example['formatted_prompt']

            # Tokenize
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                             max_length=MAX_LENGTH).to("cuda")
            num_tokens = inputs['input_ids'].shape[1]
            total_tokens += num_tokens

            # Time inference
            start = time.time()
            _ = model(**inputs)
            sample_time = time.time() - start
            sample_times.append(sample_time)

    total_time = sum(sample_times)
    tokens_per_second = total_tokens / total_time if total_time > 0 else 0
    ms_per_token = (total_time / total_tokens * 1000) if total_tokens > 0 else 0
    samples_per_second = len(samples) / total_time if total_time > 0 else 0

    return {
        'tokens_per_second': tokens_per_second,
        'ms_per_token': ms_per_token,
        'samples_per_second': samples_per_second,
        'total_time': total_time,
    }


def compute_parameter_efficiency(model, num_experts_per_tok=1):
    """
    Calculate parameter efficiency metrics.

    Returns:
        - total_params: Total model parameters
        - active_params: Parameters used per forward pass
        - trainable_params: Parameters being trained
        - sparsity_ratio: Fraction of params used (active/total)
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Check if MoE
    is_moe = False
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                break
    except:
        pass

    if is_moe:
        # Calculate active params correctly for MoE
        # Active = attention params + (k/n * expert params)
        # where k = experts per token, n = total experts
        
        # Get total expert parameters across all layers
        total_expert_params = 0
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                # Sum all expert FFN parameters in this layer
                if hasattr(layer.mlp, 'gate_proj') and isinstance(layer.mlp.gate_proj, list):
                    for expert_idx in range(num_experts):
                        total_expert_params += sum(p.numel() for p in layer.mlp.gate_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.up_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.down_proj[expert_idx].parameters())
        
        # Active expert params = (k/n) * total expert params
        sparsity = num_experts_per_tok / num_experts
        active_expert_params = int(total_expert_params * sparsity)
        
        # Non-expert params (attention, embeddings, etc)
        non_expert_params = total_params - total_expert_params
        
        # Total active = all non-expert params + active expert params
        active_params = non_expert_params + active_expert_params
    else:
        active_params = total_params

    sparsity_ratio = active_params / total_params if total_params > 0 else 1.0

    return {
        'total_params': total_params,
        'active_params': active_params,
        'trainable_params': trainable_params,
        'sparsity_ratio': sparsity_ratio,
    }


def compute_memory_metrics(model):
    """
    Compute memory usage statistics.

    Returns:
        - model_size_mb: Model size in memory
        - gpu_memory_allocated_gb: GPU memory allocated
        - gpu_memory_reserved_gb: GPU memory reserved
    """
    # Model size
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    model_size_mb = (param_size + buffer_size) / 1024 / 1024

    # GPU memory
    if torch.cuda.is_available():
        gpu_memory_allocated_gb = torch.cuda.memory_allocated() / 1024 / 1024 / 1024
        gpu_memory_reserved_gb = torch.cuda.memory_reserved() / 1024 / 1024 / 1024
    else:
        gpu_memory_allocated_gb = 0
        gpu_memory_reserved_gb = 0

    return {
        'model_size_mb': model_size_mb,
        'gpu_memory_allocated_gb': gpu_memory_allocated_gb,
        'gpu_memory_reserved_gb': gpu_memory_reserved_gb,
    }

## 12. Training Callback Reference

Note: The ComprehensiveEvalCallback class is defined below for reference, but is not used in baseline evaluation. It was designed for training-time evaluation.

In [14]:
from transformers import TrainerCallback
from torch.utils.data import DataLoader
import gc

class ComprehensiveEvalCallback(TrainerCallback):
    """
    Custom callback that evaluates and logs all Priority 1-3 metrics every N steps.
    """

    def __init__(self, eval_dataset_for_accuracy, eval_tokenized_for_loss,
                 tokenizer, answer_tokens, eval_steps=50,
                 accuracy_samples=200, device="cuda"):
        super().__init__()
        self.eval_dataset_for_accuracy = eval_dataset_for_accuracy
        self.eval_tokenized_for_loss = eval_tokenized_for_loss
        self.tokenizer = tokenizer
        self.answer_tokens = answer_tokens
        self.eval_steps = eval_steps
        self.accuracy_samples = accuracy_samples
        self.device = device

        self.metrics_history = []
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

        print(f"Evaluating every {self.eval_steps} steps")

        return control

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.eval_steps == 0 and state.global_step > 0:
            self._evaluate_and_log(args, state, model)
        return control

    def on_epoch_end(self, args, state, control, model=None, **kwargs):

        print(f"END OF EPOCH {state.epoch:.0f}")

        self._evaluate_and_log(args, state, model, is_epoch_end=True)
        return control

    def _evaluate_and_log(self, args, state, model, is_epoch_end=False):
        """Compute and log all metrics."""

        # Get training loss from state
        train_loss = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'loss' in log:
                    train_loss = log['loss']
                    break

        # Get learning rate
        learning_rate = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'learning_rate' in log:
                    learning_rate = log['learning_rate']
                    break

        # Compute evaluation loss and throughput
        eval_loss, eval_throughput, eval_latency = self._compute_eval_loss(model)

        # Compute perplexity
        perplexity = torch.exp(torch.tensor(eval_loss)).item()

        # Comprehensive MMLU evaluation
        mmlu_results = evaluate_mmlu_comprehensive(
            model=model,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset_for_accuracy,
            answer_tokens=self.answer_tokens,
            device=self.device,
            max_samples=self.accuracy_samples,
            show_progress=False
        )

        # GPU memory
        peak_gpu_memory = self._get_peak_gpu_memory()

        # Calculate elapsed time
        elapsed = time.time() - self.start_time

        # Gradient norm (if available)
        grad_norm = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'grad_norm' in log:
                    grad_norm = log['grad_norm']
                    break

        # Store metrics
        metrics = {
            'step': state.global_step,
            'epoch': state.epoch,
            'train_loss': train_loss,
            'eval_loss': eval_loss,
            'perplexity': perplexity,
            'learning_rate': learning_rate,
            'grad_norm': grad_norm,
            # MMLU metrics
            'mmlu_accuracy': mmlu_results['accuracy'],
            'mmlu_top2_accuracy': mmlu_results['top2_accuracy'],
            'mmlu_ece': mmlu_results['ece'],
            'mmlu_correct': mmlu_results['correct'],
            'mmlu_total': mmlu_results['total'],
            # Performance metrics
            'throughput': mmlu_results['throughput'],
            'avg_latency': mmlu_results['avg_latency'],
            'eval_throughput': eval_throughput,
            'eval_latency': eval_latency,
            'peak_gpu_memory_gb': peak_gpu_memory,
            'elapsed_time': elapsed,
        }
        self.metrics_history.append(metrics)

        # Log to wandb
        wandb.log({
            'step': state.global_step,
            'epoch': state.epoch,
            'train/loss': train_loss,
            'eval/loss': eval_loss,
            'eval/perplexity': perplexity,
            'train/learning_rate': learning_rate,
            'train/grad_norm': grad_norm,
            'mmlu/accuracy': mmlu_results['accuracy'],
            'mmlu/top2_accuracy': mmlu_results['top2_accuracy'],
            'mmlu/ece': mmlu_results['ece'],
            'performance/throughput': mmlu_results['throughput'],
            'performance/latency': mmlu_results['avg_latency'],
            'system/peak_gpu_memory_gb': peak_gpu_memory,
        })

        # Print metrics
        step_info = f"Epoch {state.epoch:.2f}" if is_epoch_end else f"Step {state.global_step}"

        print(f"EVALUATION at {step_info}")
        print(f"  Train Loss:      {train_loss:.4f}" if train_loss else "  Train Loss:      N/A")
        print(f"  Eval Loss:       {eval_loss:.4f}")
        print(f"  MMLU Accuracy:   {mmlu_results['accuracy']:.4f} ({mmlu_results['correct']}/{mmlu_results['total']})")
        print(f"  Perplexity:      {perplexity:.2f}")
        print(f"  Throughput:      {mmlu_results['throughput']:.2f} samples/sec")
        print(f"  Avg Latency:     {mmlu_results['avg_latency']:.4f} sec/sample")
        print(f"  Peak GPU Memory: {peak_gpu_memory:.2f} GB")
        print(f"  ECE (Calibration): {mmlu_results['ece']:.4f}")
        print(f"  Top-2 Accuracy:  {mmlu_results['top2_accuracy']:.4f}")
        if learning_rate:
            print(f"  Learning Rate:   {learning_rate:.2e}")
        if grad_norm:
            print(f"  Gradient Norm:   {grad_norm:.4f}")

        print(f" Elapsed Time:    {elapsed/60:.1f} min")


        # Clean up
        torch.cuda.empty_cache()
        gc.collect()
        model.train()

    def _compute_eval_loss(self, model, num_batches=50):
        """Compute average loss on evaluation set."""
        model.eval()
        total_loss = 0
        num_samples = 0

        eval_dataloader = DataLoader(
            self.eval_tokenized_for_loss,
            batch_size=1,
            shuffle=False
        )

        start_time = time.time()

        with torch.no_grad():
            for i, batch in enumerate(eval_dataloader):
                if i >= num_batches:
                    break

                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = model(**batch)
                total_loss += outputs.loss.item()
                num_samples += 1

        end_time = time.time()
        duration = end_time - start_time

        avg_loss = total_loss / num_samples if num_samples > 0 else 0
        throughput = num_samples / duration if duration > 0 else 0.0
        latency = duration / num_samples if num_samples > 0 else 0.0

        return avg_loss, throughput, latency

    def _get_peak_gpu_memory(self):
        """Get peak GPU memory and reset stats."""
        if torch.cuda.is_available():
            max_mem = torch.cuda.max_memory_allocated() / (1024**3)
            torch.cuda.reset_peak_memory_stats()
            return max_mem
        return 0.0

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time

        print(f"TRAINING COMPLETED in {elapsed/60:.1f} minutes")

        # Print summary table
        if self.metrics_history:
            print("TRAINING METRICS SUMMARY:")
            print(f"{'Step':<8} {'Epoch':<7} {'Train':<8} {'Eval':<8} {'Perp':<7} {'MMLU':<8} {'Top-2':<8} {'ECE':<7} {'Latency':<9}")
            print(f"{'':8} {'':7} {'Loss':<8} {'Loss':<8} {'':7} {'Acc':<8} {'Acc':<8} {'':7} {'(sec)':<9}")
            print("" * 80)
            for m in self.metrics_history:
                train_loss_str = f"{m['train_loss']:.4f}" if m['train_loss'] else "N/A"
                print(f"{m['step']:<8} {m['epoch']:<7.2f} {train_loss_str:<8} {m['eval_loss']:<8.4f} "
                      f"{m['perplexity']:<7.2f} {m['mmlu_accuracy']:<8.4f} {m['mmlu_top2_accuracy']:<8.4f} "
                      f"{m['mmlu_ece']:<7.4f} {m['avg_latency']:<9.4f}")

        return control


def evaluate_mmlu_batched(model, tokenizer, eval_dataset, answer_tokens, device="cuda",
                          batch_size=8, max_samples=None):
    """
    Batched MMLU evaluation for faster inference.
    """
    model.eval()
    samples = eval_dataset
    if max_samples is not None:
        samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    answer_token_ids = torch.tensor([answer_tokens[c] for c in ["A","B","C","D"]]).to(device)
    correct = top2_correct = 0
    total = 0

    def collate(batch):
        prompts = [ex["formatted_prompt"] for ex in batch]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        return inputs, batch

    loader = DataLoader(samples, batch_size=batch_size, shuffle=False, collate_fn=collate)

    with torch.no_grad():
        for inputs, batch in loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            last_logits = outputs.logits[:, -1, :]                 # [B, V]
            answer_logits = last_logits[:, answer_token_ids]       # [B, 4]
            probs = torch.softmax(answer_logits, dim=-1)           # [B, 4]

            pred_idx = probs.argmax(dim=-1)
            top2 = probs.topk(2, dim=-1).indices

            for i, ex in enumerate(batch):
                true_idx = answer_to_idx[ex["answer"]]
                total += 1
                correct += int(pred_idx[i].item() == true_idx)
                top2_correct += int(true_idx in top2[i])

    accuracy = correct / total if total else 0.0
    top2_accuracy = top2_correct / total if total else 0.0
    return {"accuracy": accuracy, "top2_accuracy": top2_accuracy, "correct": correct, "total": total}


print(" Comprehensive evaluation callback with batched eval defined")


 Comprehensive evaluation callback with batched eval defined


## 13. Baseline Evaluation

Evaluate the base model to establish baseline performance.

In [15]:

print("BASELINE EVALUATION WITH COMPREHENSIVE METRICS")


# 1. MMLU Accuracy
baseline_results = evaluate_mmlu_comprehensive(
    model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

# 2. FLOPs estimation
print("\nComputing FLOPs...")
baseline_flops = compute_model_flops(base_model, seq_length=MAX_LENGTH)

# 3. Throughput metrics
print("Measuring throughput...")
baseline_throughput = compute_throughput_metrics(
    model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

# 4. Parameter efficiency
print("Analyzing parameter efficiency...")
baseline_params = compute_parameter_efficiency(
    model=base_model,
    num_experts_per_tok=1  # Dense model
)

# 5. Memory metrics
print("Collecting memory metrics...")
baseline_memory = compute_memory_metrics(base_model)

# Combine all metrics
baseline_comprehensive = {
    **baseline_results,
    'flops': baseline_flops,
    **baseline_throughput,
    **baseline_params,
    **baseline_memory,
}

# Display comprehensive results

print("COMPREHENSIVE BASELINE METRICS")


print("Accuracy Metrics:")
print(f"  MMLU Accuracy: {baseline_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {baseline_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {baseline_comprehensive['ece']:.4f}")

print("\n Computational Efficiency:")
print(f"  FLOPs per forward pass: {baseline_comprehensive['flops']/1e9:.2f}G")
print(f"  Tokens/second: {baseline_comprehensive['tokens_per_second']:.2f}")
print(f"  ms/token: {baseline_comprehensive['ms_per_token']:.2f}")
print(f"  Samples/second: {baseline_comprehensive['samples_per_second']:.2f}")

print("\n Parameter Efficiency:")
print(f"  Total parameters: {baseline_comprehensive['total_params']/1e9:.2f}B")
print(f"  Active parameters: {baseline_comprehensive['active_params']/1e9:.2f}B")
print(f"  Trainable parameters: {baseline_comprehensive['trainable_params']/1e6:.2f}M")
print(f"  Sparsity ratio: {baseline_comprehensive['sparsity_ratio']:.2%}")

print("\n Memory Usage:")
print(f"  Model size: {baseline_comprehensive['model_size_mb']:.2f} MB")
print(f"  GPU allocated: {baseline_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
print(f"  GPU reserved: {baseline_comprehensive['gpu_memory_reserved_gb']:.2f} GB")

# Save comprehensive results
import json
import os
os.makedirs("results", exist_ok=True)
with open("results/baseline_comprehensive.json", 'w') as f:
    json.dump({k: v for k, v in baseline_comprehensive.items()
               if not isinstance(v, np.ndarray)}, f, indent=2, default=str)

# Log to wandb with detailed metrics
try:
    if wandb.run is not None:
        wandb.log({
            'baseline/accuracy': baseline_comprehensive['accuracy'],
            'baseline/flops_billions': baseline_comprehensive['flops']/1e9,
            'baseline/tokens_per_second': baseline_comprehensive['tokens_per_second'],
            'baseline/ms_per_token': baseline_comprehensive['ms_per_token'],
            'baseline/active_params_billions': baseline_comprehensive['active_params']/1e9,
            'baseline/gpu_memory_gb': baseline_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

# Store for later comparison
baseline_accuracy = baseline_comprehensive['accuracy']

print("\n Comprehensive baseline evaluation complete!")

BASELINE EVALUATION WITH COMPREHENSIVE METRICS


Evaluating: 100%|██████████| 1000/1000 [02:00<00:00,  8.29it/s]



Computing FLOPs...
Measuring throughput...
Analyzing parameter efficiency...
COMPREHENSIVE BASELINE METRICS
Accuracy Metrics:
  MMLU Accuracy: 0.5990
  Top-2 Accuracy: 0.7810
  ECE: 0.0541

 Computational Efficiency:
  FLOPs per forward pass: 8796.09G
  Tokens/second: 2987.88
  ms/token: 0.33
  Samples/second: 10.51

 Parameter Efficiency:
  Total parameters: 3.75B
  Active parameters: 3.75B
  Trainable parameters: 262.41M
  Sparsity ratio: 100.00%

 Memory Usage:
  Model size: 3828.51 MB
  GPU allocated: 3.87 GB
  GPU reserved: 6.88 GB

 Comprehensive baseline evaluation complete!


## 13.2 Knowledge Distillation Configuration

Define KD teacher model (dense baseline) and two stable KD configurations for MoE training.

In [16]:
# Teacher = dense baseline
teacher_model = base_model.eval()
teacher_baseline_results = baseline_comprehensive.copy()
print(f" Teacher model set (dense). Teacher accuracy: {teacher_baseline_results['accuracy']:.4f}")

# KD config 1: Output-only distillation (stable)
KD_CONFIG_STANDARD = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': False,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.001,
    'name': 'Standard KD',
}

# KD config 2: Output + light router constraints (MoE-stable)
KD_CONFIG_ROUTER_STABLE = {
    'kd_alpha': 0.6,
    'temperature': 5.0,
    'routing_kd_weight': 0.1,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': True,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.01,
    'name': 'Router-Stable KD',
}

print("KD configs ready (Standard, Router-Stable)")

 Teacher model set (dense). Teacher accuracy: 0.5990
KD configs ready (Standard, Router-Stable)


## Description

* Define router analysis functions.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [17]:
def collect_router_statistics(model, eval_dataset, tokenizer, answer_tokens,
                               max_samples=500, device="cuda"):
    """
    Collect router statistics from MoE model during inference.

    Args:
        model: MoE model
        eval_dataset: Dataset to evaluate on
        tokenizer: Tokenizer
        answer_tokens: Answer token IDs
        max_samples: Number of samples to analyze
        device: Device to run on

    Returns:
        Dict with router statistics
    """
    model.eval()

    # Detect MoE config
    first_moe_layer = None
    for layer in model.model.layers:
        if hasattr(layer.mlp, 'num_experts'):
            first_moe_layer = layer.mlp
            break

    if first_moe_layer:
        num_experts = first_moe_layer.num_experts
        num_experts_per_tok = first_moe_layer.num_experts_per_tok
    else:
        num_experts = globals().get('NUM_EXPERTS', 8)
        num_experts_per_tok = globals().get('NUM_EXPERTS_PER_TOK', 2)

    # Enable router logits collection
    for layer in model.model.layers:
        if hasattr(layer.mlp, 'forward'):
            layer.mlp._collect_router_logits = True

    # Storage for router statistics
    router_stats = {
        'expert_selections': defaultdict(lambda: np.zeros(num_experts)),
        'expert_confidence': defaultdict(list),
        'per_layer_selection': [np.zeros(num_experts) for _ in range(len(model.model.layers))],
        'per_subject_routing': defaultdict(lambda: defaultdict(lambda: np.zeros(num_experts))),
    }

    samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    print(f"Collecting router statistics from {len(samples)} samples (Experts: {num_experts}, k: {num_experts_per_tok})...")

    with torch.no_grad():
        for example in tqdm(samples, desc="Collecting router stats"):
            prompt = example['formatted_prompt']
            subject = example.get('subject', 'default')

            # Tokenize
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            # Forward pass
            outputs = model(**inputs)

            # Collect router information from each layer
            for layer_idx, layer in enumerate(model.model.layers):
                if hasattr(layer.mlp, '_last_router_probs') and layer.mlp._last_router_probs is not None:
                    router_probs = layer.mlp._last_router_probs.cpu().numpy()

                    # Average across all tokens in sequence
                    avg_probs = router_probs.mean(axis=0)

                    # Track which experts are selected (top-k)
                    top_experts = np.argsort(avg_probs)[-num_experts_per_tok:]

                    # Update statistics
                    router_stats['per_layer_selection'][layer_idx] += avg_probs

                    for expert_idx in top_experts:
                        router_stats['expert_selections']['overall'][expert_idx] += 1
                        router_stats['per_subject_routing'][subject][layer_idx][expert_idx] += 1

                    # Track confidence (probability of top expert)
                    router_stats['expert_confidence'][layer_idx].append(avg_probs.max())

    # Normalize statistics
    for layer_idx in range(len(router_stats['per_layer_selection'])):
        total = router_stats['per_layer_selection'][layer_idx].sum()
        if total > 0:
            router_stats['per_layer_selection'][layer_idx] /= total

    # Disable router collection
    for layer in model.model.layers:
        if hasattr(layer.mlp, '_collect_router_logits'):
            layer.mlp._collect_router_logits = False

    print("Router statistics collected!")
    return router_stats


def visualize_router_statistics(router_stats, title="MoE Router Analysis"):
    """
    Create comprehensive visualizations of router behavior.

    Args:
        router_stats: Router statistics from collect_router_statistics
        title: Title for the analysis
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Dynamically detect num_experts from data
    expert_usage = router_stats['expert_selections']['overall']
    num_experts = len(expert_usage)

    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # 1. Expert utilization across all layers
    ax1 = fig.add_subplot(gs[0, :2])
    if expert_usage.sum() > 0:
        expert_usage_norm = expert_usage / expert_usage.sum()
    else:
        expert_usage_norm = expert_usage

    bars = ax1.bar(range(num_experts), expert_usage_norm, color='steelblue', alpha=0.7)
    ax1.axhline(1/num_experts, color='red', linestyle='--', label='Uniform distribution')
    ax1.set_xlabel('Expert ID', fontsize=12)
    ax1.set_ylabel('Selection Frequency', fontsize=12)
    ax1.set_title('Overall Expert Utilization (All Layers)', fontsize=14, fontweight='bold')
    ax1.set_xticks(range(num_experts))
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Add percentage labels on bars
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height*100:.1f}%', ha='center', va='bottom', fontsize=9)

    # 2. Expert utilization heatmap across layers
    ax2 = fig.add_subplot(gs[0, 2])
    layer_expert_matrix = np.array(router_stats['per_layer_selection'])
    sns.heatmap(layer_expert_matrix.T, cmap='YlOrRd', ax=ax2, cbar_kws={'label': 'Selection Prob'})
    ax2.set_xlabel('Layer', fontsize=10)
    ax2.set_ylabel('Expert ID', fontsize=10)
    ax2.set_title('Expert Selection Heatmap\n(Layer vs Expert)', fontsize=12, fontweight='bold')

    # 3. Load balance score per layer
    ax3 = fig.add_subplot(gs[1, 0])
    load_balance_scores = []
    for layer_probs in router_stats['per_layer_selection']:
        # Compute entropy as measure of load balance
        if layer_probs.sum() > 0:
            layer_probs_norm = layer_probs / layer_probs.sum()
            entropy = -np.sum(layer_probs_norm * np.log(layer_probs_norm + 1e-10))
            normalized_entropy = entropy / np.log(num_experts)
        else:
            normalized_entropy = 0
        load_balance_scores.append(normalized_entropy)

    ax3.plot(load_balance_scores, marker='o', linewidth=2, markersize=4)
    ax3.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect balance')
    ax3.axhline(0.5, color='orange', linestyle='--', alpha=0.5, label='50% balance')
    ax3.set_xlabel('Layer', fontsize=10)
    ax3.set_ylabel('Load Balance Score', fontsize=10)
    ax3.set_title('Load Balancing Across Layers\n(1.0 = perfect balance)', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(alpha=0.3)
    ax3.set_ylim([0, 1.1])

    # 4. Router confidence distribution
    ax4 = fig.add_subplot(gs[1, 1])
    all_confidences = []
    for layer_confs in router_stats['expert_confidence'].values():
        all_confidences.extend(layer_confs)

    ax4.hist(all_confidences, bins=50, color='purple', alpha=0.7, edgecolor='black')
    ax4.axvline(np.mean(all_confidences), color='red', linestyle='--',
                linewidth=2, label=f"Mean: {np.mean(all_confidences):.3f}")
    ax4.set_xlabel('Router Confidence (Max Prob)', fontsize=10)
    ax4.set_ylabel('Frequency', fontsize=10)
    ax4.set_title('Distribution of Router Confidence', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)

    # 5. Expert specialization: variance across layers
    ax5 = fig.add_subplot(gs[1, 2])
    expert_variances = []
    for expert_id in range(num_experts):
        expert_across_layers = layer_expert_matrix[:, expert_id]
        variance = np.var(expert_across_layers)
        expert_variances.append(variance)

    ax5.bar(range(num_experts), expert_variances, color='teal', alpha=0.7)
    ax5.set_xlabel('Expert ID', fontsize=10)
    ax5.set_ylabel('Variance Across Layers', fontsize=10)
    ax5.set_title('Expert Specialization\n(Higher = more layer-specific)', fontsize=12, fontweight='bold')
    ax5.set_xticks(range(num_experts))
    ax5.grid(axis='y', alpha=0.3)

    # 6. Per-layer confidence box plots
    ax6 = fig.add_subplot(gs[2, :])
    layer_confidence_data = [router_stats['expert_confidence'][i]
                             for i in range(len(router_stats['expert_confidence']))]
    bp = ax6.boxplot(layer_confidence_data, patch_artist=True, showmeans=True)

    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)

    ax6.set_xlabel('Layer', fontsize=12)
    ax6.set_ylabel('Router Confidence', fontsize=12)
    ax6.set_title('Router Confidence Distribution Across Layers', fontsize=14, fontweight='bold')
    ax6.grid(axis='y', alpha=0.3)

    plt.suptitle(title, fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('router_visualization.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to 'router_visualization.png'")
    plt.show()

    # Print statistics
  
    print("ROUTER STATISTICS SUMMARY")
 

    print(f"Average load balance score: {np.mean(load_balance_scores):.4f}")
    print(f"Average router confidence: {np.mean(all_confidences):.4f}")
    print(f"Std dev of expert usage: {np.std(expert_usage_norm):.4f}")

    # Check for router collapse
    max_expert_usage = expert_usage_norm.max()
    if max_expert_usage > 0.3:
        print(f"\nWARNING: Potential router collapse detected!")
        print(f"   Expert {expert_usage_norm.argmax()} is selected {max_expert_usage*100:.1f}% of the time")
    else:
        print(f"\nNo router collapse detected (max usage: {max_expert_usage*100:.1f}%)")


## 14. MoE (Mixture of Experts) Baseline Implementation

Transform the dense Mistral-7B model into a sparse MoE model following Mixtral 8×7B architecture:
- Replace dense FFN layers with MoE layers (8 experts, top-2 routing)
- Implement sparse upcycling: duplicate FFN weights to initialize experts
- Add router with load balancing loss
- Maintain 4-bit quantization and baseline evaluation format


### 15.1 Model Architecture Modification

Replace dense FFN layers in Mistral model with MoE layers. This preserves:
- Sliding Window Attention (SWA)
- Grouped Query Attention (GQA)  
- 4-bit quantization
- All other model components


In [23]:
import torch
import torch.nn as nn
from typing import Optional

class MoELayer(nn.Module):
    """
    Mixture of Experts layer with sparse routing (top-k selection).
    
    MEMORY-OPTIMIZED with CPU OFFLOADING:
    - Experts are kept on CPU by default
    - Only active experts (selected by router) are moved to GPU during forward pass
    - After computation, experts are moved back to CPU
    - This reduces GPU memory from ~45GB to ~11GB (for top-2 out of 8 experts)

    Args:
        hidden_size: Model hidden dimension
        intermediate_size: FFN intermediate dimension per expert
        num_experts: Total number of experts
        num_experts_per_tok: Number of experts to activate per token (k in top-k)
        router_jitter_noise: Noise for router stability
        router_aux_loss_coef: Coefficient for auxiliary load balancing loss
        bnb_config: BitsAndBytes quantization config (optional)
        device: Device for router and computations (experts use CPU offloading)
        init_on_cpu: If True, initialize experts on CPU first (memory efficient)
        dtype: Data type for non-quantized experts (default: torch.bfloat16)
        enable_cpu_offload: If True, keep experts on CPU and move to GPU on-demand
    """

    def __init__(self, hidden_size, intermediate_size, num_experts=8,
                 num_experts_per_tok=2, router_jitter_noise=0.0,
                 router_aux_loss_coef=0.001, bnb_config=None, device="cuda",
                 init_on_cpu=True, dtype=torch.bfloat16, enable_cpu_offload=True):
        super().__init__()

        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.router_jitter_noise = router_jitter_noise
        self.router_aux_loss_coef = router_aux_loss_coef
        self.bnb_config = bnb_config
        self.device = device
        self.dtype = dtype
        # FIXED: CPU offloading breaks gradients for trainable (non-quantized) experts!
        # Only enable for quantized models where experts are frozen
        if bnb_config is None and enable_cpu_offload:
            print("  WARNING: Disabling CPU offload for non-quantized experts (breaks gradients)")
            enable_cpu_offload = False
        self.enable_cpu_offload = enable_cpu_offload

        # Configure linear layer class and arguments
        if bnb_config is not None:
            from bitsandbytes.nn import Linear4bit  # OPTION 2: Changed from Linear8bitLt
            LinearClass = Linear4bit
            self.compute_dtype = torch.bfloat16  # 4-bit uses bfloat16 for compute

            linear_kwargs = {
                'device': device,
                'compute_dtype': bnb_config.bnb_4bit_compute_dtype if hasattr(bnb_config, 'bnb_4bit_compute_dtype') else torch.bfloat16,
                'quant_type': bnb_config.bnb_4bit_quant_type if hasattr(bnb_config, 'bnb_4bit_quant_type') else 'nf4',
                'compress_statistics': bnb_config.bnb_4bit_use_double_quant if hasattr(bnb_config, 'bnb_4bit_use_double_quant') else False,
            }
            init_device = device  # Linear4bit needs to be on device
            self.enable_cpu_offload = False  # Disable for quantized (incompatible)
        else:
            LinearClass = nn.Linear
            self.compute_dtype = dtype
            # NEW: Always init on CPU when using CPU offloading
            init_device = 'cpu' if (init_on_cpu or enable_cpu_offload) else device
            linear_kwargs = {'device': init_device, 'dtype': dtype}

        # Determine router dtype: use float16 for quantized models, otherwise use model dtype
        # Router needs to match hidden_states dtype (float16) not compute_dtype (float32)
        # FIXED: Use bfloat16 for router to match quantized model output dtype
        router_dtype = torch.bfloat16 if bnb_config is not None else dtype

        # Initialize experts (gate_proj, up_proj, down_proj)
        # These will stay on CPU when enable_cpu_offload=True
        self.gate_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.up_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.down_proj = nn.ModuleList([
            LinearClass(intermediate_size, hidden_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])

        # Router: maps hidden state to expert logits (small, always on GPU)
        # FIXED: Initialize router to prefer expert 0 (which has perfect pretrained weights)
        self.router = nn.Linear(hidden_size, num_experts, bias=False, device=device, dtype=router_dtype)
        with torch.no_grad():
            # Initialize so expert 0 gets high probability initially
            self.router.weight.zero_()
            # OPTION 4: Enhanced initialization for quantized models
            # OPTION 4: Stronger bias for quantized models (2.0 vs 1.0)
            # Quantized activations have different distributions, need stronger signal
            expert0_bias = 2.0 if bnb_config is not None else 1.0
            self.router.weight[0, :] = expert0_bias
            # OPTION 4: Smaller noise for other experts (0.005 vs 0.01)
            # Reduces interference with expert 0 preference
            if num_experts > 1:
                noise_scale = 0.005 if bnb_config is not None else 0.01
                self.router.weight[1:, :] = torch.randn(num_experts-1, hidden_size, device=device, dtype=router_dtype) * noise_scale

        # For tracking router statistics
        self._last_router_probs = None
        self._collect_router_logits = False

        # NEW: Track which experts are currently on GPU (for CPU offloading)
        self._experts_on_gpu = set()

    def _move_expert_to_device(self, expert_idx, target_device):
        """
        Move a specific expert to target device (GPU or CPU).
        Handles gradient preservation for training.
        """
        # Move all three projections for this expert
        # Use in-place operations to preserve the module in ModuleList
        self.gate_proj[expert_idx].to(target_device)
        self.up_proj[expert_idx].to(target_device)
        self.down_proj[expert_idx].to(target_device)

    def forward(self, hidden_states):
        # Save original dtype to restore at the end
        original_dtype = hidden_states.dtype
        
        # Reshape hidden states if needed
        hidden_states_reshaped = hidden_states.view(-1, hidden_states.size(-1))  # [B*S, H]

        # Cast to router dtype for routing computation
        router_input = hidden_states_reshaped.to(self.router.weight.dtype)
        router_logits = self.router(router_input)  # [B*S, num_experts]

        if self.training and self.router_jitter_noise > 0:
            router_logits = router_logits + torch.normal(
                0, self.router_jitter_noise, size=router_logits.shape, device=router_logits.device
            )

        router_probs = torch.softmax(router_logits, dim=-1)
        top_k_probs, top_k_indices = torch.topk(router_probs, self.num_experts_per_tok, dim=-1)  # [B*S, k]

        # Store router probs for aux loss
        if self.training:
            self._last_router_probs = router_probs.view(hidden_states.size(0), hidden_states.size(1), -1)

        # Create output tensor with original dtype
        output = torch.zeros_like(hidden_states_reshaped)

        # Process each expert
        for expert_idx in range(self.num_experts):
            # Find tokens assigned to this expert
            expert_mask = (top_k_indices == expert_idx).any(dim=-1)  # [B*S]
            if expert_mask.sum() == 0:
                continue

            # Get expert weights for this token
            expert_weights = (top_k_indices == expert_idx).float()  # [B*S, k]
            expert_weights = expert_weights * top_k_probs  # Weight by probability
            expert_weights = expert_weights.sum(dim=-1, keepdim=True)  # [B*S, 1]

            # Apply expert FFN
            expert_hidden = hidden_states_reshaped[expert_mask]  # [num_tokens, H]

            # Cast input to computation dtype if needed
            if self.compute_dtype is not None and expert_hidden.dtype != self.compute_dtype:
                expert_hidden = expert_hidden.to(dtype=self.compute_dtype)

            # Gate projection (SiLU activation like Mistral)
            gate_out = torch.nn.functional.silu(self.gate_proj[expert_idx](expert_hidden))
            up_out = self.up_proj[expert_idx](expert_hidden)
            expert_out = gate_out * up_out  # [num_tokens, I]
            expert_out = self.down_proj[expert_idx](expert_out)  # [num_tokens, H]

            # Cast output back to output tensor dtype
            expert_out = expert_out.to(dtype=output.dtype)

            # Add to output (weighted by expert probability)
            output[expert_mask] += expert_weights[expert_mask] * expert_out

        # Reshape output and ensure it matches original input dtype
        output = output.view_as(hidden_states)
        return output.to(original_dtype)

    def compute_auxiliary_loss(self):
        """
        Compute auxiliary load balancing loss.
        L_aux = sum(D_i * P_i) where D_i = expert frequency, P_i = router probability
        """
        if self._last_router_probs is None:
            return torch.tensor(0.0, device=self.router.weight.device)

        # Expert frequency (how often each is selected)
        expert_freq = self._last_router_probs.mean(dim=[0, 1])  # [num_experts]

        # Router confidence (average probability assigned)
        router_confidence = self._last_router_probs.mean(dim=[0, 1])  # [num_experts]

        # Auxiliary loss: minimize correlation
        aux_loss = torch.sum(expert_freq * router_confidence) * self.num_experts

        return aux_loss * self.router_aux_loss_coef

print(" MoELayer with CPU OFFLOADING defined successfully")
print("  - Experts stay on CPU, moved to GPU only when needed")
print("  - Expected memory reduction: ~75-85% (GPU memory for experts)")



 MoELayer with CPU OFFLOADING defined successfully
  - Experts stay on CPU, moved to GPU only when needed
  - Expected memory reduction: ~75-85% (GPU memory for experts)


### 15.2 Create MoE Model

Apply sparse upcycling to transform the baseline model into MoE architecture.


In [24]:
def replace_ffn_with_moe(model, num_experts=8, num_experts_per_tok=2,
                         router_jitter_noise=0.0, router_aux_loss_coef=0.001,
                         bnb_config=None, ram_threshold=50.0, use_disk_offload=True,
                         layer_indices=None, half_width=False, enable_cpu_offload=True):
    """
    Replace dense FFN layers with MoE layers.
    IMPROVED VERSION with efficient expert dispatch, optional intermediate_size/2,
    and support for partial layer replacement.
    
    Memory-optimized: Creates MoE layers on CPU, copies weights one expert at a time,
    then moves each expert to GPU individually to avoid OOM.
    
    Args:
        model: The model to convert
        num_experts: Number of experts (default: 8)
        num_experts_per_tok: Number of experts to activate per token (default: 2)
        router_jitter_noise: Noise for router stability (default: 0.0)
        router_aux_loss_coef: Load balancing loss coefficient (default: 0.001)
        bnb_config: BitsAndBytes quantization config
        ram_threshold: RAM usage threshold for cleanup (default: 50.0)
        use_disk_offload: Whether to use disk offloading (default: True)
        layer_indices: List of layer indices to replace (default: None = all layers)
        half_width: If True, use intermediate_size // 2 for memory efficiency (default: False)
        enable_cpu_offload: If True, keep experts on CPU and move to GPU on-demand (default: True)
    """
    import gc
    import tempfile
    import os
    import psutil
    from bitsandbytes.nn import Linear4bit as BnbLinear  # OPTION 2: Changed from Linear8bitLt

    # Set CUDA allocator config to reduce fragmentation
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

    config = model.config
    hidden_size = config.hidden_size
    
    # Use half-width experts if specified (saves ~50% memory)
    if half_width:
        intermediate_size = config.intermediate_size // 2
        print(f"   Using HALF-WIDTH experts (intermediate_size // 2)")
    else:
        intermediate_size = config.intermediate_size

    print(f"Model configuration:")
    print(f"  Hidden size: {hidden_size}")
    print(f"  Original intermediate size: {config.intermediate_size}")
    print(f"  MoE intermediate size: {intermediate_size}")
    print(f"  Number of layers: {config.num_hidden_layers}")
    print(f"  Experts per layer: {num_experts}")
    print(f"  Experts per token: {num_experts_per_tok}")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available!")

    device = torch.device("cuda")
    # Handle both 4-bit and 8-bit configs
    if bnb_config and hasattr(bnb_config, 'bnb_4bit_compute_dtype'):
        compute_dtype = bnb_config.bnb_4bit_compute_dtype  # 4-bit config
    elif bnb_config and hasattr(bnb_config, 'llm_int8_threshold'):
        compute_dtype = torch.bfloat16  # 8-bit config
    else:
        compute_dtype = torch.bfloat16  # Default

    print(f"\n Using GPU for weight processing")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Compute dtype: {compute_dtype}")

    # Helper functions
    def check_ram():
        return psutil.virtual_memory().percent

    def cleanup(aggressive=False):
        gc.collect()
        torch.cuda.empty_cache()
        if aggressive:
            torch.cuda.synchronize()
        try:
            import ctypes
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except:
            pass
        gc.collect()

    def print_memory_stats(label=""):
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"    [{label}] GPU: {allocated:.2f}GB alloc / {reserved:.2f}GB reserved")

    def extract_weight(linear_layer, expected_shape, keep_on_cpu=True):
        """Extract and dequantize weight from a layer."""
        # #region agent log
        try:
            import json, time; log_data = {"id":"log_extract_weight_entry","timestamp":int(time.time()*1000),"location":"extract_weight:entry","message":"extract_weight called","data":{"expected_shape":str(expected_shape),"expected_shape_type":str(type(expected_shape).__name__),"has_numel":hasattr(expected_shape,'numel')},"sessionId":"debug-session","runId":"post-fix","hypothesisId":"A"}; open('/root/shit/.cursor/debug.log','a').write(json.dumps(log_data)+'\n')
        except: pass
        # #endregion
        from bitsandbytes.nn import Linear4bit as BnbLinear  # OPTION 2: Changed from Linear8bitLt  # Import here for nested function scope
        # Calculate expected numel from tuple/shape - FIX: expected_shape is a tuple, not a tensor
        expected_numel = torch.Size(expected_shape).numel() if isinstance(expected_shape, (tuple, list)) else expected_shape.numel()
        # #region agent log
        try:
            log_data = {"id":"log_expected_numel_calc","timestamp":int(time.time()*1000),"location":"extract_weight:numel_calc","message":"calculated expected_numel","data":{"expected_shape":str(expected_shape),"expected_numel":int(expected_numel)},"sessionId":"debug-session","runId":"post-fix","hypothesisId":"A"}; open('/root/shit/.cursor/debug.log','a').write(json.dumps(log_data)+'\n')
        except: pass
        # #endregion
        if isinstance(linear_layer, BnbLinear):  # Works for both Linear4bit and Linear8bitLt
            # 8-bit weights can be accessed directly (no dequantization needed)
            weight = linear_layer.weight.data.to(compute_dtype)
            # #region agent log
            try:
                log_data = {"id":"log_weight_extracted","timestamp":int(time.time()*1000),"location":"extract_weight:weight_extracted","message":"weight extracted from BnbLinear","data":{"weight_shape":str(weight.shape),"weight_numel":int(weight.numel())},"sessionId":"debug-session","runId":"post-fix","hypothesisId":"A"}; open('/root/shit/.cursor/debug.log','a').write(json.dumps(log_data)+'\n')
            except: pass
            # #endregion
            # Ensure proper shape - handle flattened or transposed weights
            if weight.shape != expected_shape:
                if weight.numel() == expected_numel:
                    # Use reshape instead of view for better compatibility
                    weight = weight.reshape(expected_shape)
                elif len(weight.shape) == 2 and len(expected_shape) == 2:
                    # Check if transposed
                    if weight.shape[0] == expected_shape[1] and weight.shape[1] == expected_shape[0]:
                        weight = weight.t()
                    elif weight.numel() == expected_numel:
                        weight = weight.reshape(expected_shape)
            return weight.cpu() if keep_on_cpu else weight.cuda()
        else:
            weight = linear_layer.weight.data.to(compute_dtype)
            # #region agent log
            try:
                log_data = {"id":"log_weight_extracted_non_bnb","timestamp":int(time.time()*1000),"location":"extract_weight:weight_extracted_non_bnb","message":"weight extracted from non-BnbLinear","data":{"weight_shape":str(weight.shape),"weight_numel":int(weight.numel())},"sessionId":"debug-session","runId":"post-fix","hypothesisId":"A"}; open('/root/shit/.cursor/debug.log','a').write(json.dumps(log_data)+'\n')
            except: pass
            # #endregion
            # Ensure shape matches for non-quantized layers too
            if weight.shape != expected_shape and weight.numel() == expected_numel:
                weight = weight.reshape(expected_shape)
            return weight.cpu() if keep_on_cpu else weight

    # Determine layers to process
    total_layers = len(model.model.layers)
    if layer_indices is None:
        target_layers = list(range(total_layers))
    else:
        target_layers = layer_indices
    print(f"\n  Processing {len(target_layers)} layers: {target_layers[:5]}{'...' if len(target_layers) > 5 else ''}\n")

    for i in target_layers:
        layer = model.model.layers[i]
        original_mlp = layer.mlp

        ram = check_ram()
        print(f"  Layer {i+1}/{total_layers} (RAM: {ram:.1f}%):")

        # Step 1: Extract all weights to CPU (memory efficient)
        print(f"    Extracting weights to CPU...", end=" ", flush=True)
        with torch.no_grad():
            # Extract weights and keep on CPU
            gate_w_full = extract_weight(original_mlp.gate_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            up_w_full = extract_weight(original_mlp.up_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            down_w_full = extract_weight(original_mlp.down_proj, (hidden_size, config.intermediate_size), keep_on_cpu=True)

            # Slice to target intermediate size if needed
            gate_w = gate_w_full[:intermediate_size, :].clone()
            up_w = up_w_full[:intermediate_size, :].clone()
            down_w = down_w_full[:, :intermediate_size].clone()

            # Delete full weights IMMEDIATELY
            del gate_w_full, up_w_full, down_w_full
            gc.collect()
        print("Done")

        # Step 2: Delete original MLP to free GPU memory
        print(f"    Deleting original MLP...", end=" ", flush=True)
        del original_mlp
        layer.mlp = None
        cleanup(aggressive=True)
        print("Done")

        # Step 3: Create MoE layer on CPU (memory efficient)
        print(f"    Creating MoE layer on CPU...", end=" ", flush=True)
        moe_layer = MoELayer(
            hidden_size=hidden_size,
            intermediate_size=intermediate_size,
            num_experts=num_experts,
            num_experts_per_tok=num_experts_per_tok,
            router_jitter_noise=router_jitter_noise,
            router_aux_loss_coef=router_aux_loss_coef,
            bnb_config=bnb_config,
            device='cpu' if bnb_config is None else device,  # CPU for non-quantized
            init_on_cpu=(bnb_config is None),  # Initialize on CPU when not using quantization
            dtype=compute_dtype,  # Use bfloat16 from the start to avoid dtype mismatch
            enable_cpu_offload=enable_cpu_offload  # NEW: Enable CPU offloading for memory efficiency
        )
        print("Done")

        # Step 4: Copy weights to all experts (on CPU) first
        print(f"    Copying weights to {num_experts} experts...", end=" ", flush=True)
        with torch.no_grad():
            # Validate and fix shapes before copying
            gate_target_shape = moe_layer.gate_proj[0].weight.shape
            up_target_shape = moe_layer.up_proj[0].weight.shape
            down_target_shape = moe_layer.down_proj[0].weight.shape
            
            # Fix gate_w shape if needed
            if gate_w.shape != gate_target_shape:
                if gate_w.numel() == gate_target_shape.numel():
                    gate_w = gate_w.reshape(gate_target_shape)
                else:
                    gate_w = gate_w[:gate_target_shape[0], :gate_target_shape[1]]
            
            # Fix up_w shape if needed
            if up_w.shape != up_target_shape:
                if up_w.numel() == up_target_shape.numel():
                    up_w = up_w.reshape(up_target_shape)
                else:
                    up_w = up_w[:up_target_shape[0], :up_target_shape[1]]
            
            # Fix down_w shape if needed - this is the problematic one
            if down_w.shape != down_target_shape:
                if down_w.numel() == down_target_shape.numel():
                    down_w = down_w.reshape(down_target_shape)
                elif len(down_w.shape) == 1:
                    # If flattened, reshape to target
                    down_w = down_w.reshape(down_target_shape)
                elif down_w.shape[0] == down_target_shape[0] and down_w.shape[1] >= down_target_shape[1]:
                    down_w = down_w[:, :down_target_shape[1]]
                elif down_w.shape[1] == down_target_shape[1] and down_w.shape[0] >= down_target_shape[0]:
                    down_w = down_w[:down_target_shape[0], :]
                else:
                    raise RuntimeError(
                        f"Cannot reshape down_w from {down_w.shape} to {down_target_shape}. "
                        f"down_w.numel()={down_w.numel()}, target.numel()={down_target_shape.numel()}, "
                        f"down_w.size(0)={down_w.size(0) if len(down_w.shape) > 0 else 'N/A'}, "
                        f"target.size(0)={down_target_shape[0]}"
                    )
            
            for idx in range(num_experts):
                # Copy weights (on CPU) - shapes should now match
                moe_layer.gate_proj[idx].weight.copy_(gate_w)
                moe_layer.up_proj[idx].weight.copy_(up_w)
                moe_layer.down_proj[idx].weight.copy_(down_w)

                # FIXED: Only add TINY noise to non-primary experts to break symmetry
                # Expert 0 keeps perfect pretrained weights (no noise)
                if idx > 0:
                    # Add minimal noise (0.1% vs old 5%) to preserve pretrained knowledge
                    moe_layer.gate_proj[idx].weight.data += torch.randn_like(moe_layer.gate_proj[idx].weight) * 0.001
                    moe_layer.up_proj[idx].weight.data += torch.randn_like(moe_layer.up_proj[idx].weight) * 0.001
                    moe_layer.down_proj[idx].weight.data += torch.randn_like(moe_layer.down_proj[idx].weight) * 0.001
        print("Done")
        
        # Delete extracted weights BEFORE moving to GPU
        del gate_w, up_w, down_w
        gc.collect()

        # Step 5: Move entire ModuleLists to GPU at once (faster than one-by-one)
        print(f"    Moving to GPU...", end=" ", flush=True)
        moe_layer.gate_proj = moe_layer.gate_proj.to(device)
        moe_layer.up_proj = moe_layer.up_proj.to(device)
        moe_layer.down_proj = moe_layer.down_proj.to(device)
        moe_layer.router = moe_layer.router.to(device)
        print("Done")

        # Step 6: Replace and cleanup
        layer.mlp = moe_layer
        cleanup(aggressive=True)

        # Progress update every 4 layers
        if (i + 1) % 4 == 0:
            print_memory_stats(f"Layer {i+1}")

    cleanup(aggressive=True)

    print(f"\n Successfully replaced {len(target_layers)} FFN layers with MoE")
    print(f"  Expert dispatch: Efficient sparse routing")
    print(f"  CPU offloading: {'ENABLED' if enable_cpu_offload else 'DISABLED'} (keeps experts on CPU, moves to GPU on-demand)")
    print(f"  Params per expert: ~{(intermediate_size * hidden_size * 3) / 1e6:.1f}M")
    print(f"  Active params per token: ~{(intermediate_size * hidden_size * 3 * num_experts_per_tok) / 1e6:.1f}M")

    # Final stats
    gpu_final = torch.cuda.memory_allocated() / 1e9
    ram_final = check_ram()
    print(f"  Final: GPU {gpu_final:.2f}GB | RAM {ram_final:.1f}%")
    return model


print(" Improved replace_ffn_with_moe with memory optimizations:")
print("  - Batch GPU transfers (faster)")
print("  - Aggressive garbage collection")
print("  - Early deletion of intermediate tensors")
print("  - CUDA allocator config to reduce fragmentation")

 Improved replace_ffn_with_moe with memory optimizations:
  - Batch GPU transfers (faster)
  - Aggressive garbage collection
  - Early deletion of intermediate tensors
  - CUDA allocator config to reduce fragmentation


### 15.3 Auxiliary Load Balancing Loss

Implement the auxiliary loss function for load balancing across experts.


In [25]:
def compute_moe_auxiliary_loss(model, router_aux_loss_coef=0.001):
    """
    Compute auxiliary load balancing loss from all MoE layers.
    L_aux = sum(D_i * P_i) where:
    - D_i = expert frequency (actual utilization rate)
    - P_i = router probability (router confidence)

    Args:
        model: MoE model
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        aux_loss: Auxiliary loss value
    """
    total_aux_loss = torch.tensor(0.0, device=next(model.parameters()).device)

    # Collect router information from all MoE layers
    for layer in model.model.layers:
        if hasattr(layer.mlp, '_last_router_probs') and layer.mlp._last_router_probs is not None:
            router_probs = layer.mlp._last_router_probs

            # Expert frequency: how often each expert is selected (average probability)
            expert_freq = router_probs.mean(dim=0)  # (num_experts,)

            # Router confidence: average probability assigned to each expert
            router_confidence = router_probs.mean(dim=0)  # (num_experts,)

            # Auxiliary loss: minimize correlation between frequency and confidence
            layer_aux_loss = torch.sum(expert_freq * router_confidence) * layer.mlp.num_experts
            total_aux_loss = total_aux_loss + layer_aux_loss

    return total_aux_loss

def compute_moe_loss(model, outputs, router_aux_loss_coef=0.001):
    """
    Compute total loss for MoE model: L_total = L_NTP + λ * L_aux

    Args:
        model: MoE model
        outputs: Model outputs with loss
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        total_loss: Combined loss
        ntp_loss: Next token prediction loss
        aux_loss: Auxiliary load balancing loss
    """
    # Get the standard next token prediction loss
    ntp_loss = outputs.loss if hasattr(outputs, 'loss') else None

    # Compute auxiliary loss from all MoE layers
    aux_loss = compute_moe_auxiliary_loss(model, router_aux_loss_coef)

    # Total loss
    if ntp_loss is not None:
        total_loss = ntp_loss + router_aux_loss_coef * aux_loss
    else:
        total_loss = router_aux_loss_coef * aux_loss

    return total_loss, ntp_loss, aux_loss

print("Auxiliary loss computation function defined")


Auxiliary loss computation function defined


### 15.4 MoE Model Evaluation

Evaluate the MoE model on MMLU using the same format as the baseline .


## MoE Training Strategy

Key improvements to enable expert specialization:

1. **Noise Initialization**: Each expert receives small random perturbations (stddev=0.01) during initialization to break symmetry
2. **Unfrozen Experts**: Expert FFN parameters are trainable (not just LoRA adapters)
3. **Corrected Parameter Counting**: Active params calculated as attention + (k/n * expert_params)

This allows:
- Experts to differentiate and specialize during training
- Router to learn meaningful routing decisions
- Accurate efficiency metrics


In [27]:
# ============================================================================
# MoE CONFIGURATION - MEMORY OPTIMIZED (FIXED FOR 80GB GPU)
# ============================================================================
# FIXED: Use 4-bit quantized experts to match MoEExperimentRunner approach
# - Reduces memory from 116 GB → 25 GB (78% reduction)
# - Prevents kernel crashes on 80 GB GPU
# - Same architecture as Runner for consistent comparison
# Memory: 8 experts × 32 layers × 4-bit quantization ≈ 25 GB
# ============================================================================

NUM_EXPERTS = 8  # Full 8 experts like Mixtral
NUM_EXPERTS_PER_TOK = 2  # Top-2 routing like Mixtral
ROUTER_JITTER_NOISE = 0.0
ROUTER_AUX_LOSS_COEF = 0.001  # FIXED: Reduced from 0.01 to allow expert 0 preference initially
USE_HALF_WIDTH = False  # FIXED: Use full intermediate_size (quantization makes this safe)


print("MoE Configuration ")

print(f"  Number of experts: {NUM_EXPERTS}")
print(f"  Experts per token: {NUM_EXPERTS_PER_TOK}")
print(f"  Half-width experts: {USE_HALF_WIDTH} (intermediate_size // 2)")
print(f"  Router jitter noise: {ROUTER_JITTER_NOISE}")
print(f"  Router aux loss coefficient: {ROUTER_AUX_LOSS_COEF}")

# Estimate memory with quantization
intermediate = 14336 // 2 if USE_HALF_WIDTH else 14336  # Full width
expert_params = NUM_EXPERTS * 32 * 3 * 4096 * intermediate  # experts × layers × projections × dims
expert_memory_bf16 = expert_params * 2 / 1e9  # bf16 = 2 bytes
expert_memory_4bit = expert_params * 0.5 / 1e9  # 4-bit = 0.5 bytes
print(f"\nMemory Estimate:")
print(f"  Intermediate size: {intermediate} {'(half-width)' if USE_HALF_WIDTH else '(full)'}")
print(f"  Expert params: {expert_params/1e9:.2f}B")
print(f"  Expert memory (bf16): ~{expert_memory_bf16:.1f} GB  ← Old approach (too large!)")
print(f"  Expert memory (4-bit): ~{expert_memory_4bit:.1f} GB  ← New approach (safe!)")

# ============================================================================
# AGGRESSIVE MEMORY CLEANUP BEFORE MOE CREATION
# ============================================================================
import gc
import torch

print("\n" + "="*70)
print("MEMORY CLEANUP BEFORE MOE CREATION")
print("="*70)

# Check initial memory
gpu_initial = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory before cleanup: {gpu_initial:.2f} GB")

# Delete base_model if it exists and is not the teacher
if 'base_model' in globals() and 'teacher_model' in globals():
    if base_model is not teacher_model:
        print("  Deleting base_model (keeping teacher_model for KD)")
        del base_model
    else:
        print("  base_model IS teacher_model, keeping it")
elif 'base_model' in globals():
    print("  Deleting base_model")
    del base_model

# Delete any other large model variables
vars_to_delete = ['model']
deleted_count = 0
for var_name in list(globals().keys()):
    if var_name in vars_to_delete and var_name in globals():
        obj = globals().get(var_name)
        if obj is not None and hasattr(obj, 'parameters'):
            try:
                next(obj.parameters())
                print(f"  Deleting: {var_name}")
                del globals()[var_name]
                deleted_count += 1
            except:
                pass

# Aggressive cleanup
for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()
torch.cuda.synchronize()

# Try to trigger malloc_trim
try:
    import ctypes
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except:
    pass

gpu_after_cleanup = torch.cuda.memory_allocated() / 1e9
print(f"\nGPU memory after cleanup: {gpu_after_cleanup:.2f} GB")
print(f"Freed: {gpu_initial - gpu_after_cleanup:.2f} GB")

# ============================================================================
# CREATE 8-BIT QUANTIZATION CONFIG FOR EXPERTS
# ============================================================================
from transformers import BitsAndBytesConfig

print("\n" + "="*70)
print("CREATING 4-BIT QUANTIZATION CONFIG FOR EXPERTS (OPTION 2)")
print("="*70)

moe_expert_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # OPTION 2: Changed to 4-bit for better accuracy
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,  # Better accuracy
    bnb_4bit_quant_type="nf4",  # NormalFloat4 (best 4-bit method)
)

print(" Quantization config created")
print(f"  Type: 4-bit NF4 (trainable with LoRA)")
print(f"  Compute dtype: bfloat16, Double quantization: Enabled (better accuracy)")
print(f"  Expected reduction: 90.2 GB → 22.5 GB (75% less)")
print(f"  Trainable: Yes (with LoRA adapters)")

# ============================================================================
# LOAD FRESH BASE MODEL FOR MOE CONVERSION
# ============================================================================
print("\n" + "="*70)
print("LOADING FRESH MODEL FOR MOE CONVERSION")
print("="*70)

from transformers import AutoModelForCausalLM

# Use same config as original base model
base_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # OPTION 2: Changed to 4-bit for consistency
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading fresh Mistral-7B model...")
moe_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=base_bnb_config,  # Base model 4-bit
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    )

gpu_with_base = torch.cuda.memory_allocated() / 1e9
print(f" Fresh model loaded")
print(f"  GPU memory: {gpu_with_base:.2f} GB")

# ============================================================================
# CREATE MOE WITH 4-BIT QUANTIZED EXPERTS (OPTION 2)
# ============================================================================
print("\n" + "="*70)
print("CONVERTING TO MOE WITH 4-BIT EXPERTS")
print("="*70)

print(f"Creating MoE model...")
print(f"  - {NUM_EXPERTS} experts, top-{NUM_EXPERTS_PER_TOK} routing")
print(f"  - Expert precision: 4-bit quantized")
print(f"  - Intermediate size: {intermediate}")
print(f"  - CPU offload: Disabled (breaks gradients)")

moe_model = replace_ffn_with_moe(
    model=moe_model,
    num_experts=NUM_EXPERTS,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK,
    router_jitter_noise=ROUTER_JITTER_NOISE,
    router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
    bnb_config=moe_expert_bnb_config,  # OPTION 2: 4-bit quantized experts (was 8-bit)
    ram_threshold=80.0,                 #  Enable cleanup at 80% (was 0.0)
    use_disk_offload=True,
    half_width=USE_HALF_WIDTH,
    enable_cpu_offload=False,           # Disabled (breaks training gradients)
)

# Final cleanup
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

gpu_final = torch.cuda.memory_allocated() / 1e9
gpu_reserved = torch.cuda.memory_reserved() / 1e9
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\n{'='*70}")
print(" MOE MODEL CREATED SUCCESSFULLY")
print(f"{'='*70}")
print(f"  GPU Memory:")
print(f"    Allocated: {gpu_final:.2f} GB")
print(f"    Reserved:  {gpu_reserved:.2f} GB")
print(f"    Total:     {gpu_total:.2f} GB")
print(f"    Usage:     {100*gpu_final/gpu_total:.1f}%")
print(f"")
print(f"  Expected: ~25-30 GB (vs 115 GB before fix)")
print(f"  Safe for 80 GB GPU: {'YES ' if gpu_final < 70 else 'NO '}")
print(f"")
print(f"  Configuration:")
print(f"    - Experts: {NUM_EXPERTS} ({NUM_EXPERTS_PER_TOK} active per token)")
print(f"    - Expert precision: 4-bit quantized")
print(f"    - Attention layers: 4-bit quantized")
print(f"    - Router: bf16 (trainable)")
print(f"    - Load balancing coefficient: {ROUTER_AUX_LOSS_COEF}")
print(f"{'='*70}")



MoE Configuration 
  Number of experts: 8
  Experts per token: 2
  Half-width experts: False (intermediate_size // 2)
  Router jitter noise: 0.0
  Router aux loss coefficient: 0.001

Memory Estimate:
  Intermediate size: 14336 (full)
  Expert params: 45.10B
  Expert memory (bf16): ~90.2 GB  ← Old approach (too large!)
  Expert memory (4-bit): ~22.5 GB  ← New approach (safe!)

MEMORY CLEANUP BEFORE MOE CREATION
GPU memory before cleanup: 12.38 GB
  base_model IS teacher_model, keeping it

GPU memory after cleanup: 8.19 GB
Freed: 4.18 GB

CREATING 4-BIT QUANTIZATION CONFIG FOR EXPERTS (OPTION 2)
 Quantization config created
  Type: 4-bit NF4 (trainable with LoRA)
  Compute dtype: bfloat16, Double quantization: Enabled (better accuracy)
  Expected reduction: 90.2 GB → 22.5 GB (75% less)
  Trainable: Yes (with LoRA adapters)

LOADING FRESH MODEL FOR MOE CONVERSION
Loading fresh Mistral-7B model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.15s/it]

 Fresh model loaded
  GPU memory: 12.32 GB

CONVERTING TO MOE WITH 4-BIT EXPERTS
Creating MoE model...
  - 8 experts, top-2 routing
  - Expert precision: 4-bit quantized
  - Intermediate size: 14336
  - CPU offload: Disabled (breaks gradients)
Model configuration:
  Hidden size: 4096
  Original intermediate size: 14336
  MoE intermediate size: 14336
  Number of layers: 32
  Experts per layer: 8
  Experts per token: 2

 Using GPU for weight processing
  GPU Memory: 150.11 GB
  Compute dtype: torch.bfloat16

  Processing 32 layers: [0, 1, 2, 3, 4]...

  Layer 1/32 (RAM: 10.8%):
    Extracting weights to CPU... 

Done
    Deleting original MLP... Done
    Creating MoE layer on CPU... 

TypeError: Linear4bit.__init__() got an unexpected keyword argument 'has_fp16_weights'

## Description

* Define memory monitoring utilities.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# ============================================================================
# MEMORY MONITORING UTILITIES FOR CPU OFFLOADING
# ============================================================================

import torch
import gc

def print_gpu_memory_summary():
    """Print detailed GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_allocated = torch.cuda.max_memory_allocated() / 1024**3
        
        print("\n" + "="*70)
        print("GPU MEMORY USAGE")
        print("="*70)
        print(f"  Currently allocated: {allocated:.2f} GB")
        print(f"  Reserved by PyTorch: {reserved:.2f} GB")
        print(f"  Peak allocated:      {max_allocated:.2f} GB")
        print("="*70)
    else:
        print("CUDA not available")

def get_expert_device_distribution(moe_model):
    """
    Check which device each expert is on (CPU vs GPU).
    Returns dict with counts of experts on each device.
    """
    device_counts = {'cuda': 0, 'cpu': 0}
    expert_locations = []
    
    for layer_idx, layer in enumerate(moe_model.model.layers):
        if hasattr(layer.mlp, 'num_experts'):
            for expert_idx in range(layer.mlp.num_experts):
                device = str(layer.mlp.gate_proj[expert_idx].weight.device)
                if 'cuda' in device:
                    device_counts['cuda'] += 1
                    device_type = 'GPU'
                else:
                    device_counts['cpu'] += 1
                    device_type = 'CPU'
                expert_locations.append({
                    'layer': layer_idx,
                    'expert': expert_idx,
                    'device': device_type
                })
    
    return device_counts, expert_locations

def verify_cpu_offloading(moe_model):
    """
    Verify that CPU offloading is working correctly.
    Prints summary of expert device placement.
    """
   
    print("CPU OFFLOADING VERIFICATION")
  
    
    device_counts, expert_locations = get_expert_device_distribution(moe_model)
    
    total_experts = device_counts['cuda'] + device_counts['cpu']
    
    print(f"\nTotal experts across all layers: {total_experts}")
    print(f"  Experts on GPU: {device_counts['cuda']} ({100*device_counts['cuda']/total_experts:.1f}%)")
    print(f"  Experts on CPU: {device_counts['cpu']} ({100*device_counts['cpu']/total_experts:.1f}%)")
    
    if device_counts['cpu'] > device_counts['cuda']:
        print("\n CPU offloading is ACTIVE - majority of experts on CPU")
    else:
        print("\n CPU offloading may not be active - most experts on GPU")
    
    # Show first layer as example
    first_layer_experts = [e for e in expert_locations if e['layer'] == 0]
    if first_layer_experts:
        print(f"\nExample - Layer 0 expert placement:")
        for exp in first_layer_experts[:8]:  # Show up to 8 experts
            print(f"  Expert {exp['expert']}: {exp['device']}")
    
    print("="*70)
    
    return device_counts

print(" Memory monitoring utilities loaded")
print("  - print_gpu_memory_summary(): Check current GPU memory")
print("  - verify_cpu_offloading(model): Check expert device placement")
print("  - get_expert_device_distribution(model): Get detailed expert locations")


## Description

* Test and verify CPU offloading functionality.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# ============================================================================
# TEST: VERIFY CPU OFFLOADING IS WORKING
# ============================================================================

print("\n" + "="*70)
print("TESTING CPU OFFLOADING FUNCTIONALITY")
print("="*70)

# Clear GPU cache first
torch.cuda.empty_cache()
gc.collect()

print("\n[1/4] Initial GPU memory state...")
print_gpu_memory_summary()

print("\n[2/4] Checking expert device placement...")
device_counts = verify_cpu_offloading(moe_model)

print("\n[3/4] Running a test forward pass to trigger expert loading...")
# Create a small test input
test_input = tokenizer("This is a test sentence for CPU offloading.", 
                       return_tensors="pt", 
                       padding=False, 
                       truncation=True).to("cuda")

# Run forward pass (this should move active experts to GPU)
with torch.no_grad():
    moe_model.eval()
    outputs = moe_model(**test_input)
    
print(" Forward pass completed successfully")

print("\n[4/4] GPU memory after forward pass...")
print_gpu_memory_summary()

print("\n" + "="*70)
print("CPU OFFLOADING TEST SUMMARY")
print("="*70)

if device_counts['cpu'] > 0:
    savings_pct = 100 * device_counts['cpu'] / (device_counts['cpu'] + device_counts['cuda'])
    print(f" SUCCESS: {savings_pct:.1f}% of experts are on CPU")
    print(f" Expected GPU memory reduction: ~{savings_pct:.0f}%")
    print(f"\n CPU offloading is ACTIVE and working correctly!")
else:
    print(" WARNING: All experts are on GPU")
    print("  CPU offloading may not be enabled.")
    print("  Check that enable_cpu_offload=True in replace_ffn_with_moe()")

print("="*70)


## Description

* Execute comprehensive baseline evaluation with all metrics.
* This cell implements the functionality described above
* Results and outputs are displayed below


## Description

* Run baseline model evaluation.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# ============================================================================
# MOE BASELINE EVALUATION (BEFORE QLoRA)
# ============================================================================
# This evaluates the pure MoE model with 4-bit quantization ONLY
# No LoRA adapters are applied yet - this is the true baseline
# ============================================================================

print("="*70)
print("MOE BASELINE EVALUATION (4-bit quantized, NO LoRA)")
print("="*70)

# 1. MMLU Accuracy
print("\n Evaluating MMLU accuracy...")
moe_results = evaluate_mmlu_comprehensive(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

# 2. FLOPs estimation
print("\n Computing FLOPs...")
moe_flops = compute_model_flops(moe_model, seq_length=MAX_LENGTH)

# 3. Throughput metrics
print(" Measuring throughput...")
moe_throughput = compute_throughput_metrics(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

# 4. Parameter efficiency
print(" Analyzing parameter efficiency...")
moe_params = compute_parameter_efficiency(
    model=moe_model,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK
)

# 5. Memory metrics
print(" Collecting memory metrics...")
moe_memory = compute_memory_metrics(moe_model)

# Combine all metrics
moe_comprehensive = {
    **moe_results,
    'flops': moe_flops,
    **moe_throughput,
    **moe_params,
    **moe_memory,
}

print("\n" + "="*70)
print("MoE Baseline Evaluation Complete (Pre-LoRA)")
print("="*70)
print(f"  Accuracy: {moe_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {moe_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {moe_comprehensive['ece']:.4f}")
print("\n Note: This is the TRUE baseline - no LoRA adapters applied yet")
print("   Next step: Apply QLoRA for training (Cell 15.5.2)")

### 15.5 Training Setup for Sparse Upcycling

Configure training infrastructure for sparse upcycling (pre-training, no SFT).
Note: This is the training setup - actual training can be run separately.


In [ ]:

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from torch.utils.data import Dataset as TorchDataset
import torch

class MoETrainer(Trainer):
    """
    Custom Trainer for MoE models that computes auxiliary load balancing loss.
    Works with both regular models and PEFT-wrapped models.
    Total loss formula: L_total = L_NTP + λ * L_aux
    """

    def __init__(self, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.router_aux_loss_coef = router_aux_loss_coef

    def _moe_layers(self, model):
        """Get MoE layers from model, handling PEFT wrapper."""
        # Try different model structures (PEFT wrapped, base model, etc.)
        candidates = [
            lambda m: m.model.layers if hasattr(m, 'model') and hasattr(m.model, 'layers') else None,
            lambda m: m.base_model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'layers') else None,
            lambda m: m.base_model.model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'model') else None,
        ]
        
        for get_layers in candidates:
            try:
                layers = get_layers(model)
                if layers is not None:
                    return layers
            except:
                continue
        return []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Enable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        # Forward pass
        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        
        # Compute auxiliary load balancing loss
        aux_loss = self._compute_moe_auxiliary_loss(model)
        total_loss = ntp_loss + self.router_aux_loss_coef * aux_loss

        # Log losses periodically
        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "train/ntp_loss": ntp_loss.item(),
                "train/aux_loss": aux_loss.item(),
                "train/total_loss": total_loss.item(),
                "train/aux_loss_weight": self.router_aux_loss_coef,
            })

        # Disable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

    def _compute_moe_auxiliary_loss(self, model):
        """Compute auxiliary loss from all MoE layers."""
        device = next(model.parameters()).device
        aux_loss = torch.tensor(0.0, device=device)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()
        return aux_loss

# Training configuration for QLoRA
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": ROUTER_AUX_LOSS_COEF,
    "learning_rate": 2e-4,  # QLoRA standard learning rate
    "batch_size": 4,  # Larger batch with QLoRA
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "output_dir": "./mistral_moe_qlora",
}

print(" MoETrainer class defined (QLoRA compatible)\n")

## Description

* Define KD trainer for MoE.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
print("MoETrainer class defined\n")

# ============================================================================
# 15.5.2B - INTEGRATED MOE KD TRAINER
# ============================================================================

import torch.nn.functional as F

class IntegratedMoEKDTrainer(Trainer):
    """
    KD for MoE: L_total = (1-α)*L_NTP + α*L_KD + λ*L_aux + β*L_routingKD
    """
    def __init__(self, teacher_model=None, kd_config=None, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.kd_config = kd_config or {}
        self.router_aux_loss_coef = router_aux_loss_coef
        self.kd_alpha = self.kd_config.get('kd_alpha', 0.5)
        self.temperature = self.kd_config.get('temperature', 4.0)
        self.routing_kd_weight = self.kd_config.get('routing_kd_weight', 0.0)
        self.enable_routing_kd = self.kd_config.get('enable_routing_kd', False)

    def _moe_layers(self, model):
        if hasattr(model, 'model') and hasattr(model.model, 'layers'):
            return model.model.layers
        if hasattr(model, 'base_model') and hasattr(model.base_model.model, 'layers'):
            return model.base_model.model.layers
        return []

    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Compute loss with KD.
        NOTE: Trainer automatically handles device placement - inputs are already on correct device!
        """
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        device = ntp_loss.device if hasattr(ntp_loss, 'device') else next(model.parameters()).device

        kd_loss = torch.tensor(0.0, device=device)
        routing_kd_loss = torch.tensor(0.0, device=device)

        if self.teacher_model is not None and self.kd_alpha > 0:
            tdev = next(self.teacher_model.parameters()).device
            tinputs = {k: (v.to(tdev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
            with torch.no_grad():
                t_out = self.teacher_model(**tinputs)

            s_logits = outputs.logits / self.temperature
            t_logits = t_out.logits / self.temperature

            kd_loss = F.kl_div(
                F.log_softmax(s_logits, dim=-1),
                F.softmax(t_logits, dim=-1),
                reduction='batchmean'
            ) * (self.temperature ** 2)

            if self.enable_routing_kd and self.routing_kd_weight > 0:
                ent = torch.tensor(0.0, device=device)
                layers_count = 0
                for layer in self._moe_layers(model):
                    mlp = getattr(layer, 'mlp', None)
                    if mlp is not None and hasattr(mlp, '_last_router_probs') and mlp._last_router_probs is not None:
                        probs = mlp._last_router_probs
                        ent_layer = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean()
                        ent = ent + ent_layer
                        layers_count += 1
                if layers_count > 0:
                    routing_kd_loss = ent / layers_count

        aux_loss = torch.tensor(0.0, device=device)
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()

        total_loss = ((1 - self.kd_alpha) * ntp_loss
                      + self.kd_alpha * kd_loss
                      + self.router_aux_loss_coef * aux_loss
                      + self.routing_kd_weight * routing_kd_loss)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

print(" IntegratedMoEKDTrainer class defined")

## Description

* Define experiment configurations.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
from dataclasses import dataclass
from typing import Literal, List, Optional
import json

@dataclass
class MoEExperimentConfig:
    """Configuration for MoE variant experiments."""

    # Basic architecture
    num_experts: int = 8
    num_experts_per_tok: int = 2

    # Routing configuration
    router_jitter_noise: float = 0.0
    router_aux_loss_coef: float = 0.001

    # Expert placement
    expert_layers: Literal["all", "every_2", "every_4", "selected"] = "all"
    layer_indices: Optional[List[int]] = None

    # Training
    load_balancing_loss_coef: float = 0.01

    # Metadata
    experiment_name: str = "default"
    description: str = ""

    def to_dict(self):
        return {
            'num_experts': self.num_experts,
            'num_experts_per_tok': self.num_experts_per_tok,
            'router_jitter_noise': self.router_jitter_noise,
            'router_aux_loss_coef': self.router_aux_loss_coef,
            'expert_layers': self.expert_layers,
            'layer_indices': self.layer_indices,
            'load_balancing_loss_coef': self.load_balancing_loss_coef,
            'experiment_name': self.experiment_name,
            'description': self.description,
        }

    def save(self, filepath):
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def load(cls, filepath):
        with open(filepath, 'r') as f:
            return cls(**json.load(f))


# Predefined experiment configurations
EXPERIMENT_CONFIGS = {
    # --- 1. Baseline & Routing Variants ---
    "baseline_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="all",
        experiment_name="baseline_8x2",
        description="Standard 8 experts, top-2 routing, all layers"
    ),

    "top1_8x1": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="top1_8x1",
        description="Top-1 routing: 8 experts, single expert per token"
    ),

    "top1_16x1": MoEExperimentConfig(
        num_experts=16,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="top1_16x1",
        description="Top-1 routing: 16 experts, single expert per token"
    ),

    "routing_noisy_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        router_jitter_noise=0.2,
        expert_layers="all",
        experiment_name="routing_noisy_8x2",
        description="Noisy routing: 8 experts, top-2, high jitter (0.2)"
    ),

    "balanced_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        router_aux_loss_coef=0.05,  # Higher coefficient for strict balancing
        expert_layers="all",
        experiment_name="balanced_8x2",
        description="Load Balanced: 8 experts, top-2, high aux loss coef (0.05)"
    ),

    # --- 2. Expert Count Variants ---
    "efficient_4x1": MoEExperimentConfig(
        num_experts=4,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="efficient_4x1",
        description="Efficient: 4 experts, top-1 routing"
    ),

    "large_16x2": MoEExperimentConfig(
        num_experts=16,
        num_experts_per_tok=2,
        expert_layers="all",
        experiment_name="large_16x2",
        description="Large: 16 experts, top-2 routing"
    ),

    # --- 3. Placement Variants ---
    "sparse_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="every_2",
        experiment_name="sparse_8x2",
        description="Sparse placement: experts every 2nd layer"
    ),

    "placement_early_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(0, 16)),  # First 16 layers
        experiment_name="placement_early_8x2",
        description="Early placement: Experts in first 16 layers only"
    ),

    "placement_middle_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(8, 24)),  # Middle 16 layers
        experiment_name="placement_middle_8x2",
        description="Middle placement: Experts in middle 16 layers (8-23)"
    ),

    "placement_late_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(16, 32)),  # Last 16 layers
        experiment_name="placement_late_8x2",
        description="Late placement: Experts in last 16 layers only"
    ),

    "placement_mixed_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=[0, 1, 2, 3, 14, 15, 16, 17, 28, 29, 30, 31],  # First 4, Middle 4, Last 4
        experiment_name="placement_mixed_8x2",
        description="Mixed placement: Experts in first 4, middle 4, and last 4 layers"
    ),
}

print(" MoE variant configuration system defined")
print(f"  Available configs: {list(EXPERIMENT_CONFIGS.keys())}")

## Description

* Load the language model with quantization.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
class MoEExperimentRunner:
    """System for running and tracking MoE variant experiments."""

    def __init__(self, base_model, tokenizer, eval_dataset, answer_tokens, train_dataset=None):
        self.base_model = base_model
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.answer_tokens = answer_tokens
        self.train_dataset = train_dataset
        self.results = {}

    def run_experiment(self, config: MoEExperimentConfig, max_samples=500, train=False, train_steps=50):
        """
        Run a complete experiment with given configuration.

        Args:
            config: Experiment configuration
            max_samples: Number of samples for evaluation (Lower = Faster)
            train: Whether to train the model
            train_steps: Number of training steps (Lower = Faster)
        """
        print(f"\n{'='*70}")
        print(f"RUNNING EXPERIMENT: {config.experiment_name}")
        print(f"{'='*70}\n")
        print(f"Description: {config.description}")
        print(f"Configuration:")
        print(f"  Experts: {config.num_experts}")
        print(f"  Experts per token: {config.num_experts_per_tok}")
        print(f"  Layer placement: {config.expert_layers}\n")

        # 1. Create model with config
        moe_model = self._create_moe_model(config)

        # 2. Pre-training Evaluation
        print(f"\nPhase 1: Pre-training Evaluation (n={max_samples})...")
        pre_results = self._evaluate_comprehensive(
            model=moe_model,
            config=config,
            max_samples=max_samples
        )

        print("\n" + "="*70)
        print("COMPREHENSIVE PRE-TRAINING METRICS")
        print("="*70 + "\n")

        print("Accuracy Metrics:")
        print(f"  MMLU Accuracy: {pre_results['accuracy']:.4f}")
        print(f"  Top-2 Accuracy: {pre_results['top2_accuracy']:.4f}")
        print(f"  ECE: {pre_results['ece']:.4f}")

        print("\nComputational Efficiency:")
        print(f"  FLOPs per forward pass: {pre_results['flops']/1e9:.2f}G")
        print(f"  Tokens/second: {pre_results['tokens_per_second']:.2f}")
        print(f"  ms/token: {pre_results['ms_per_token']:.2f}")
        print(f"  Samples/second: {pre_results['samples_per_second']:.2f}")

        print("\nParameter Efficiency:")
        print(f"  Total parameters: {pre_results['total_params']/1e9:.2f}B")
        print(f"  Active parameters: {pre_results['active_params']/1e9:.2f}B")
        print(f"  Trainable parameters: {pre_results['trainable_params']/1e6:.2f}M")
        print(f"  Sparsity ratio: {pre_results['sparsity_ratio']:.2%}")

        print("\nMemory Usage:")
        print(f"  Model size: {pre_results['model_size_mb']:.2f} MB")
        print(f"  GPU allocated: {pre_results['gpu_memory_allocated_gb']:.2f} GB")
        print(f"  GPU reserved: {pre_results['gpu_memory_reserved_gb']:.2f} GB")

        final_results = pre_results.copy()
        final_results['phase'] = 'pre_train_only'

        # 3. Training (Optional)
        if train and self.train_dataset:
            print(f"\nPhase 2: Training for {train_steps} steps...")

            # Setup LoRA and Trainer
            moe_model = self._apply_lora(moe_model)
            self._train_model(moe_model, config, train_steps)

            # 4. Post-training Evaluation
            print(f"\nPhase 3: Post-training Evaluation (n={max_samples})...")
            post_results = self._evaluate_comprehensive(
                model=moe_model,
                config=config,
                max_samples=max_samples
            )

            print("\n" + "="*70)
            print("COMPREHENSIVE POST-TRAINING METRICS")
            print("="*70 + "\n")

            print("Accuracy Metrics:")
            print(f"  MMLU Accuracy: {post_results['accuracy']:.4f}")
            print(f"  Top-2 Accuracy: {post_results['top2_accuracy']:.4f}")
            print(f"  ECE: {post_results['ece']:.4f}")

            print("\nComputational Efficiency:")
            print(f"  FLOPs per forward pass: {post_results['flops']/1e9:.2f}G")
            print(f"  Tokens/second: {post_results['tokens_per_second']:.2f}")
            print(f"  ms/token: {post_results['ms_per_token']:.2f}")
            print(f"  Samples/second: {post_results['samples_per_second']:.2f}")

            print("\nParameter Efficiency:")
            print(f"  Total parameters: {post_results['total_params']/1e9:.2f}B")
            print(f"  Active parameters: {post_results['active_params']/1e9:.2f}B")
            print(f"  Trainable parameters: {post_results['trainable_params']/1e6:.2f}M")
            print(f"  Sparsity ratio: {post_results['sparsity_ratio']:.2%}")

            print("\nMemory Usage:")
            print(f"  Model size: {post_results['model_size_mb']:.2f} MB")
            print(f"  GPU allocated: {post_results['gpu_memory_allocated_gb']:.2f} GB")
            print(f"  GPU reserved: {post_results['gpu_memory_reserved_gb']:.2f} GB")

            # Merge results for comparison
            final_results = post_results.copy()
            final_results['phase'] = 'trained'
            final_results['pre_train_accuracy'] = pre_results['accuracy']
            final_results['accuracy_gain'] = post_results['accuracy'] - pre_results['accuracy']
            final_results['pre_train_results'] = pre_results

        # Store results
        final_results['config'] = config.to_dict()
        final_results['timestamp'] = time.time()

        self.results[config.experiment_name] = final_results

        # Save results
        self._save_results(config.experiment_name, final_results)

        # Cleanup
        del moe_model
        torch.cuda.empty_cache()
        gc.collect()

        return final_results

    def _create_moe_model(self, config):
        """Create MoE model according to configuration."""
        from transformers import AutoModelForCausalLM

        # Load fresh model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )

        # Determine which layers to modify
        total_layers = len(model.model.layers)
        if config.expert_layers == "all":
            target_layers = None
        elif config.expert_layers == "every_2":
            target_layers = list(range(0, total_layers, 2))
        elif config.expert_layers == "every_4":
            target_layers = list(range(0, total_layers, 4))
        elif config.expert_layers == "selected" and config.layer_indices:
            target_layers = config.layer_indices
        else:
            target_layers = None

        # Apply MoE transformation
        model = replace_ffn_with_moe(
            model=model,
            num_experts=config.num_experts,
            num_experts_per_tok=config.num_experts_per_tok,
            router_jitter_noise=config.router_jitter_noise,
            router_aux_loss_coef=config.router_aux_loss_coef,
            bnb_config=bnb_config,
            layer_indices=target_layers,
            ram_threshold=80.0,
            use_disk_offload=True
        )

        torch.cuda.empty_cache()
        return model

    def _apply_lora(self, model):
        """Apply LoRA adapters for training."""
        from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

        model = prepare_model_for_kbit_training(model)

        lora_config = LoraConfig(
            r=16, lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05, bias="none",
            task_type=TaskType.CAUSAL_LM,
            modules_to_save=["router"]
        )

        model = get_peft_model(model, lora_config)
        model.train()
        return model

    def _train_model(self, model, config, steps):
        """Run short training loop."""
        from transformers import TrainingArguments, DataCollatorForLanguageModeling

        args = TrainingArguments(
            output_dir=f"experiments/{config.experiment_name}_checkpoints",
            max_steps=steps,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            logging_steps=10,
            bf16=True,
            optim="paged_adamw_8bit",
            save_strategy="no",
            report_to="none"
        )

        trainer = MoETrainer(
            model=model,
            args=args,
            train_dataset=self.train_dataset,
            tokenizer=self.tokenizer,
            data_collator=DataCollatorForLanguageModeling(self.tokenizer, mlm=False),
            router_aux_loss_coef=config.router_aux_loss_coef
        )

        trainer.train()

    def _evaluate_comprehensive(self, model, config, max_samples):
        """Run comprehensive evaluation."""
        results = {}

        print(f"  Running MMLU evaluation (n={max_samples})...")
        mmlu_results = evaluate_mmlu_comprehensive(
            model=model,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset,
            answer_tokens=self.answer_tokens,
            device="cuda",
            max_samples=max_samples,
            show_progress=False
        )

        # Fast FLOPs (estimate)
        flops = compute_model_flops(model, seq_length=MAX_LENGTH)

        # Fast throughput check (10 samples)
        throughput_metrics = compute_throughput_metrics(
            model=model,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset,
            max_samples=50
        )

        # Parameter stats
        param_metrics = compute_parameter_efficiency(
            model=model,
            num_experts_per_tok=config.num_experts_per_tok
        )

        # Memory stats
        memory_metrics = compute_memory_metrics(model)

        results.update({
            'accuracy': mmlu_results['accuracy'],
            'top2_accuracy': mmlu_results['top2_accuracy'],
            'ece': mmlu_results['ece'],
            'flops': flops,
            **throughput_metrics,
            **param_metrics,
            **memory_metrics,
        })

        print(f"  -> Accuracy: {results['accuracy']:.4f} | Throughput: {results['tokens_per_second']:.1f} tok/s")
        return results

    def _save_results(self, exp_name, results):
        """Save results to disk."""
        filepath = f"experiments/{exp_name}_results.json"
        os.makedirs("experiments", exist_ok=True)

        # Helper to make JSON serializable
        def make_serializable(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, np.float32):
                return float(obj)
            return str(obj)

        clean_results = {k: (v if not isinstance(v, (np.ndarray, set)) else str(v)) for k, v in results.items()}

        with open(filepath, 'w') as f:
            json.dump(clean_results, f, indent=2, default=make_serializable)

    def compare_experiments(self, exp_names=None):
        """Generate comparison table."""
        if exp_names is None:
            exp_names = list(self.results.keys())

        comparison_data = []
        for name in exp_names:
            if name in self.results:
                exp = self.results[name]

                # Format gain string if available
                gain_str = "-"
                if 'accuracy_gain' in exp:
                    gain = exp['accuracy_gain'] * 100
                    gain_str = f"{gain:+.2f}%"

                comparison_data.append({
                    'Experiment': name,
                    'Pre Acc': f"{exp.get('pre_train_accuracy', exp['accuracy']):.4f}",
                    'Post Acc': f"{exp['accuracy']:.4f}",
                    'Gain': gain_str,
                    'Top-2': f"{exp['top2_accuracy']:.4f}",
                    'ECE': f"{exp['ece']:.4f}",
                    'FLOPs (G)': f"{exp.get('flops', 0) / 1e9:.2f}",
                    'Tokens/sec': f"{exp['tokens_per_second']:.1f}",
                    'ms/token': f"{exp.get('ms_per_token', 0):.2f}",
                    'Samples/sec': f"{exp.get('samples_per_second', 0):.2f}",
                    'Total Params (B)': f"{exp['total_params']/1e9:.2f}",
                    'Active Params (B)': f"{exp['active_params']/1e9:.2f}",
                    'Trainable Params (M)': f"{exp.get('trainable_params', 0)/1e6:.2f}",
                    'Sparsity': f"{exp['sparsity_ratio']:.2%}",
                    'Model Size (MB)': f"{exp.get('model_size_mb', 0):.2f}",
                    'GPU Alloc (GB)': f"{exp['gpu_memory_allocated_gb']:.2f}",
                    'GPU Reserved (GB)': f"{exp.get('gpu_memory_reserved_gb', 0):.2f}",
                })

        if not comparison_data:
            print("No results to compare yet.")
            return pd.DataFrame()

        df = pd.DataFrame(comparison_data)
        print("\n" + "="*170)
        print("EXPERIMENT COMPARISON (FAST TRACK)")
        print("="*170 + "\n")
        print(df.to_string(index=False))

        return df

# 15.5.1 Tokenization

## Description

* Tokenize datasets for training.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
from transformers import DataCollatorForLanguageModeling

print("Preparing datasets for training...\n")

# REDUCE to 10% for faster training
DATASET_FRACTION = 0.10

# Calculate subset sizes
train_subset_size = int(len(train_dataset_raw) * DATASET_FRACTION)
eval_subset_size = int(len(eval_dataset_raw) * DATASET_FRACTION)

print(f"Original dataset sizes:")
print(f"  Train samples: {len(train_dataset_raw):,}")
print(f"  Eval samples:  {len(eval_dataset_raw):,}")

print(f"\nUsing {DATASET_FRACTION*100:.0f}% subset:")
print(f"  Train samples: {train_subset_size:,}")
print(f"  Eval samples:  {eval_subset_size:,}")
print(f"  Expected speedup: ~{1/DATASET_FRACTION:.0f}x faster\n")

# Select subset
train_dataset_subset = train_dataset_raw.select(range(train_subset_size))
eval_dataset_subset = eval_dataset_raw.select(range(eval_subset_size))

def tokenize_function(examples):
    """Tokenize for causal language modeling."""
    return tokenizer(
        examples['formatted_prompt'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

print("Tokenizing training dataset (subset)...")
train_dataset_tokenized = train_dataset_subset.map(
    tokenize_function,
    batched=True,
    batch_size=100, 
    remove_columns=train_dataset_subset.column_names,
    desc="Tokenizing train set (20%)",
)

print("Tokenizing evaluation dataset (subset)...")
eval_dataset_clm = eval_dataset_subset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=eval_dataset_subset.column_names,
    desc="Tokenizing eval set (20%)",
)

print("\nDatasets ready for training!")
print(f"  Training: {len(train_dataset_tokenized):,} samples (20%)")
print(f"  Evaluation: {len(eval_dataset_clm):,} samples (20%)")
print(f"  Columns: {train_dataset_tokenized.column_names}")

## Description

* Set up experiment runner.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# ============================================================================
# TRAINING WITH MOE EXPERIMENT RUNNER
# ============================================================================
# This approach uses the MoEExperimentRunner for systematic experimentation

print("="*80)
print("TRAINING WITH MOEEXPERIMENTRUNNER")
print("="*80)

# Initialize experiment runner
runner = MoEExperimentRunner(
    base_model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    train_dataset=train_dataset_tokenized
)

print("MoEExperimentRunner initialized")

# Define experiment configurations
experiments_to_run = [
    MoEExperimentConfig(
        experiment_name="baseline_8experts",
        description="Standard 8-expert MoE with top-2 routing",
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="all",
        router_aux_loss_coef=0.001,
    ),
    MoEExperimentConfig(
        experiment_name="sparse_4experts",
        description="Memory-efficient 4-expert configuration",
        num_experts=4,
        num_experts_per_tok=2,
        expert_layers="all",
        router_aux_loss_coef=0.001,
    ),
]

print(f"\nConfigured {len(experiments_to_run)} experiments:")
for config in experiments_to_run:
    print(f"  - {config.experiment_name}: {config.description}")

# Run experiments
print("\nStarting experiments...")
experiment_results = {}

for config in experiments_to_run:
    print(f"\n{'='*80}")
    print(f"Running Experiment: {config.experiment_name}")
    print(f"{'='*80}")
    
    results = runner.run_experiment(
        config=config,
        max_samples=1000,  # Full evaluation
        train=True,        # Enable training
        train_steps=250    # Quick training run
    )
    
    experiment_results[config.experiment_name] = results
    
    print(f"\n {config.experiment_name} complete!")
    print(f"  Pre-training accuracy:  {results['pre_training']['accuracy']:.4f}")
    print(f"  Post-training accuracy: {results['post_training']['accuracy']:.4f}")
    print(f"  Improvement: {(results['post_training']['accuracy'] - results['pre_training']['accuracy']):.4f} ({100*(results['post_training']['accuracy'] - results['pre_training']['accuracy'])/results['pre_training']['accuracy']:.1f}%)")

# Compare all experiments
print("\n" + "="*80)
print("EXPERIMENT COMPARISON")
print("="*80)

comparison_df = runner.compare_experiments()
print(comparison_df)

# Identify best configuration
best_config = max(experiment_results.items(), 
                  key=lambda x: x[1]['post_training']['accuracy'])

print(f"\n Best configuration: {best_config[0]}")
print(f"  Final accuracy: {best_config[1]['post_training']['accuracy']:.4f}")

print("\n" + "="*80)
print("MoEExperimentRunner training complete!")
print("="*80)


## Description

* Compare KD configurations.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
import pandas as pd
import torch
import gc
from copy import deepcopy

print("="*80)
print("KNOWLEDGE DISTILLATION BASELINE COMPARISON")
print("="*80)
print("\nRunning KD baseline comparison on 10% dataset with 250 training steps...")
print("Configurations:")
print("  1. No-KD: Standard supervised learning")
print("  2. KD-Standard: Output distillation (α=0.5, T=4.0)")
print("  3. KD-Router-Stable: Output + routing KD (α=0.6, T=5.0)\n")

# Storage for KD comparison results
kd_results = {}

def build_fresh_moe():
    """Build a fresh MoE model for KD training."""
    model = replace_ffn_with_moe(
        model=deepcopy(base_model),
        num_experts=NUM_EXPERTS,
        num_experts_per_tok=NUM_EXPERTS_PER_TOK,
        router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
        router_jitter_noise=ROUTER_JITTER_NOISE,
        use_sparse_upcycling=True,
    )
    return model

def apply_lora_to_model(model):
    """Apply LoRA configuration to a fresh model."""
    model = prepare_model_for_kbit_training(model)
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            r".*experts\.\d+\.gate_proj",
            r".*experts\.\d+\.up_proj",
            r".*experts\.\d+\.down_proj",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        modules_to_save=["router"],
        use_rslora=False,
    )
    model = get_peft_model(model, lora_cfg)
    model.train()
    return model

def train_kd_config(config_name, kd_config=None, use_kd=False):
    """Train a single KD configuration."""
    print(f"\n{'='*80}")
    print(f"Training: {config_name}")
    print(f"{'='*80}")
    
    # Build fresh model
    print(f"Building fresh MoE model...")
    model = build_fresh_moe()
    model = apply_lora_to_model(model)
    
    # Setup training arguments
    training_args = TrainingArguments(
        output_dir=f"./moe_kd_{config_name.lower().replace(' ', '_')}",
        max_steps=250,  # 10% dataset, 250 steps
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        weight_decay=0.01,
        logging_steps=50,
        logging_strategy="steps",
        save_steps=125,
        eval_strategy="steps",
        eval_steps=125,
        bf16=True,
        gradient_checkpointing=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        optim="paged_adamw_8bit",
        warmup_ratio=0.03,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to=[],  # Disabled to avoid WandB connection issues
        disable_tqdm=False,
        save_total_limit=1,
        seed=42,
    )
    
    # Select trainer class
    if use_kd and kd_config is not None:
        print(f"Using IntegratedMoEKDTrainer with config: {kd_config}")
        trainer = IntegratedMoEKDTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset_tokenized,
            eval_dataset=eval_dataset_clm,
            data_collator=data_collator,
            router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
            teacher_model=teacher_model,
            kd_config=kd_config,
        )
    else:
        print(f"Using standard MoETrainer (no KD)")
        trainer = MoETrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset_tokenized,
            eval_dataset=eval_dataset_clm,
            data_collator=data_collator,
            router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
        )
    
    # Train
    print(f"Starting training for {config_name}...")
    train_result = trainer.train()
    print(f" Training complete: final loss = {train_result.training_loss:.4f}")
    
    # Evaluate trained model
    print(f"\nEvaluating {config_name} on MMLU...")
    model.eval()
    eval_results = evaluate_mmlu_comprehensive(
        model=model,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        answer_tokens=ANSWER_TOKENS,
        device="cuda",
        max_samples=500,  # Reduced for speed
        show_progress=False
    )
    
    # Store results
    kd_results[config_name] = {
        'training_loss': train_result.training_loss,
        'eval_loss': train_result.metrics.get('eval_loss', 0.0),
        'accuracy': eval_results['accuracy'],
        'top2_accuracy': eval_results.get('top2_accuracy', 0.0),
        'training_steps': train_result.global_step,
    }
    
    print(f"Results for {config_name}:")
    print(f"  Training Loss: {kd_results[config_name]['training_loss']:.4f}")
    print(f"  Eval Loss: {kd_results[config_name]['eval_loss']:.4f}")
    print(f"  MMLU Accuracy: {kd_results[config_name]['accuracy']:.4f}")
    print(f"  Top-2 Accuracy: {kd_results[config_name]['top2_accuracy']:.4f}")
    
    # Cleanup
    del model, trainer, train_result
    torch.cuda.empty_cache()
    gc.collect()
    
    return kd_results[config_name]

# Run three configurations
print("\n" + "="*80)
print("CONFIGURATION 1: NO-KD BASELINE (Supervised Learning Only)")
print("="*80)
train_kd_config("No-KD Baseline", kd_config=None, use_kd=False)

print("\n" + "="*80)
print("CONFIGURATION 2: KD-STANDARD (Output Distillation)")
print("="*80)
train_kd_config("KD-Standard", kd_config=KD_CONFIG_STANDARD, use_kd=True)

print("\n" + "="*80)
print("CONFIGURATION 3: KD-ROUTER-STABLE (Output + Routing KD)")
print("="*80)
train_kd_config("KD-Router-Stable", kd_config=KD_CONFIG_ROUTER_STABLE, use_kd=True)

# Create comparison DataFrame
print("\n" + "="*80)
print("KD BASELINE COMPARISON RESULTS")
print("="*80)

kd_comparison_df = pd.DataFrame(kd_results).T
kd_comparison_df = kd_comparison_df.reset_index().rename(columns={'index': 'Configuration'})

print("\n" + kd_comparison_df.to_string(index=False))

# Analysis
print("\n" + "="*80)
print("ANALYSIS")
print("="*80)

baseline_acc = kd_results['No-KD Baseline']['accuracy']
standard_acc = kd_results['KD-Standard']['accuracy']
router_acc = kd_results['KD-Router-Stable']['accuracy']

print(f"\nAccuracy Improvements vs No-KD Baseline ({baseline_acc:.4f}):")
print(f"  KD-Standard: {(standard_acc - baseline_acc)*100:+.2f}% ")
print(f"  KD-Router-Stable: {(router_acc - baseline_acc)*100:+.2f}%")

if standard_acc > baseline_acc or router_acc > baseline_acc:
    print("\n KD showed improvement! Best config: " + 
          ("KD-Standard" if standard_acc >= router_acc else "KD-Router-Stable"))
else:
    print("\n  KD did not improve accuracy in this setting.")
    print("  Consider: longer training, different KD parameters, or alternative configurations")

# Save results
kd_results_path = 'results/moe_kd_baseline_comparison.json'
import json
with open(kd_results_path, 'w') as f:
    json.dump(kd_results, f, indent=2)
print(f"\n Results saved to {kd_results_path}")

# Save DataFrame
csv_path = 'results/moe_kd_baseline_comparison.csv'
kd_comparison_df.to_csv(csv_path, index=False)
print(f" Comparison CSV saved to {csv_path}")

print("\n" + "="*80)
print("KD BASELINE COMPARISON COMPLETE")
print("="*80)


# 15.5.2 Training Setup for MoE with Trainable Experts

**Your Hypothesis**: Training expert weights leads to better accuracy improvement than frozen experts.

**Training Strategy:**

| Component | Status | Why |
|-----------|--------|-----|
| **Attention layers** | LoRA adapters (trainable) | Memory efficient |
| **Router** | Fully trainable | Learns expert selection |
| **Expert weights** | **TRAINABLE (bf16)** | Your hypothesis! |

**Memory Trade-off:**
- Experts are bf16 instead of 4-bit → ~4x more memory for expert weights
- But allows full gradient updates for expert specialization

In [ ]:
# ============================================================================
# VERIFY TRAINING CONFIGURATION
# ============================================================================

import torch

print("="*70)
print("TRAINING CONFIGURATION VERIFICATION")
print("="*70)

# Count parameters by component
expert_trainable = 0
expert_frozen = 0
router_trainable = 0
lora_trainable = 0
other_frozen = 0

for name, param in moe_model.named_parameters():
    numel = param.numel()
    
    if any(x in name for x in ["gate_proj", "up_proj", "down_proj"]):
        if param.requires_grad:
            expert_trainable += numel
        else:
            expert_frozen += numel
    elif "router" in name.lower():
        if param.requires_grad:
            router_trainable += numel
    elif "lora" in name.lower():
        if param.requires_grad:
            lora_trainable += numel
    else:
        if not param.requires_grad:
            other_frozen += numel

total_trainable = expert_trainable + router_trainable + lora_trainable
total_params = sum(p.numel() for p in moe_model.parameters())

print("\nParameter Breakdown:")
print(f"  Expert weights (TRAINABLE): {expert_trainable/1e9:.2f}B ← YOUR HYPOTHESIS")
print(f"  Expert weights (frozen):    {expert_frozen/1e9:.2f}B")
print(f"  Router (trainable):         {router_trainable/1e3:.1f}K")
print(f"  Attention LoRA (trainable): {lora_trainable/1e6:.2f}M")
print(f"  Other (frozen):             {other_frozen/1e9:.2f}B")

print(f"\nTotals:")
print(f"  Total trainable: {total_trainable/1e9:.2f}B ({100*total_trainable/total_params:.1f}%)")
print(f"  Total frozen:    {(total_params-total_trainable)/1e9:.2f}B")

if expert_trainable > 0:
    print("\n" + "="*70)
    print("[OK] EXPERT WEIGHTS ARE TRAINABLE!")
    print("="*70)
    print("Your hypothesis can be tested: experts will learn during training.")
else:
    print("\n" + "="*70)
    print("WARNING: Expert weights are NOT trainable")
    print("="*70)
    print("Re-run the MoE creation cell with bnb_config=None for experts")

## Description

* Verify MoE architecture.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# ============================================================================
# VERIFY MOE ARCHITECTURE & EXPERT TRAINABILITY
# ============================================================================

import torch

print("="*70)
print("MOE ARCHITECTURE & EXPERT TRAINABILITY CHECK")
print("="*70)

# Navigate through PEFT wrapper to find MoE layers
def get_moe_layers(model):
    """Get MoE layers from model (handles PEFT wrapper)."""
    if hasattr(model, 'base_model'):
        if hasattr(model.base_model, 'model'):
            if hasattr(model.base_model.model, 'model'):
                return model.base_model.model.model.layers
            return model.base_model.model.layers
    if hasattr(model, 'model'):
        return model.model.layers
    return []

layers = get_moe_layers(moe_model)
print(f"Found {len(layers)} decoder layers")

# Find first MoE layer
first_moe_layer = None
for layer in layers:
    if hasattr(layer.mlp, "gate_proj") and isinstance(layer.mlp.gate_proj, torch.nn.ModuleList):
        first_moe_layer = layer.mlp
        break

if first_moe_layer:
    num_experts = len(first_moe_layer.gate_proj)
    print(f"\n[OK] MoE layer found with {num_experts} experts")
    print(f"  Experts per token (k): {first_moe_layer.num_experts_per_tok}")
    
    # Check expert weight type and trainability
    expert0 = first_moe_layer.gate_proj[0]
    weight_type = type(expert0).__name__
    weight_dtype = expert0.weight.dtype
    weight_trainable = expert0.weight.requires_grad
    
    print(f"\nExpert Configuration:")
    print(f"  Layer type: {weight_type}")
    print(f"  Weight dtype: {weight_dtype}")
    print(f"  Trainable: {weight_trainable}")
    
    if 'Linear4bit' in weight_type:
        print("\n  [X] 4-bit quantized - NOT trainable")
        print("     Re-run MoE creation with bnb_config=None")
    elif weight_trainable:
        print("\n  [OK] bf16/fp16 - TRAINABLE (your hypothesis enabled!)")
    else:
        print("\n  [!] Float weights but not trainable - check unfreezing step")
    
    # Check router
    print(f"\nRouter:")
    print(f"  Shape: {first_moe_layer.router.weight.shape}")
    print(f"  Dtype: {first_moe_layer.router.weight.dtype}")
    print(f"  Trainable: {first_moe_layer.router.weight.requires_grad}")
    
    # Check expert weight diversity
    print(f"\nExpert Weight Diversity:")
    try:
        w0 = first_moe_layer.gate_proj[0].weight.data.float()
        w1 = first_moe_layer.gate_proj[1].weight.data.float()
        
        diff = torch.abs(w0 - w1).mean().item()
        print(f"  Mean diff between Expert 0 and 1: {diff:.6f}")
        
        if diff > 0.001:
            print("  [OK] Experts have different weights (good for specialization)")
        else:
            print("  [!] Experts have identical weights")
    except Exception as e:
        print(f"  Could not compare: {e}")

else:
    print("[X] No MoE layer found!")

print("\n" + "="*70)

# 15.5.3 QLoRA Training

Train the MoE model using QLoRA:
- **Memory efficient**: 4-bit weights + 8-bit optimizer states
- **Higher batch size**: 4x batch size possible due to memory savings
- **Higher learning rate**: 2e-4 (standard for LoRA vs 1e-5 for full fine-tuning)

In [ ]:
# Replace the TrainingArguments cell with this MEMORY-OPTIMIZED version
# ============================================================================
# SETUP TRAINING ARGUMENTS - OPTIMIZED FOR TRAINABLE BF16 EXPERTS
# ============================================================================

from transformers import TrainingArguments
import os

# Critical: Configure CUDA allocator for large model training
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'

OUTPUT_DIR = "./mistral_moe_qlora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=500,
    num_train_epochs=1,

    # CRITICAL: Minimum batch size
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,  # Effective batch = 16

    # Learning rate
    learning_rate=2e-4,
    weight_decay=0.01,

    # Reduce evaluation frequency (eval uses memory)
    logging_steps=50,
    logging_strategy="steps",
    save_steps=500,  # Save only at end
    eval_strategy="no",  # DISABLE eval during training to save memory

    # CRITICAL Memory optimizations
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Disable dataloader parallelism
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    # 8-bit optimizer (critical for large trainable params)
    optim="paged_adamw_8bit",
    
    # Warmup
    warmup_ratio=0.03,

    # Don't load best model (saves memory)
    load_best_model_at_end=False,

    # Disable reporting during training
    report_to=[],
    disable_tqdm=False,
    save_total_limit=1,
    seed=42,
    
    # Additional memory savings
    ddp_find_unused_parameters=False,
    torch_compile=False,
    
    # CRITICAL: Enable memory-efficient attention if available
    # (This is handled by the model, but setting explicitly)
)

print("\n" + "="*70)
print("MEMORY-OPTIMIZED TRAINING FOR TRAINABLE EXPERTS")
print("="*70)
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Gradient checkpointing: ENABLED")
print(f"  Optimizer: paged_adamw_8bit")
print(f"  Evaluation during training: DISABLED (to save memory)")
print(f"  Max steps: {training_args.max_steps}")

## Description

* Create training DataLoader.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
from torch.utils.data import DataLoader
import torch

optimizer = torch.optim.AdamW(moe_model.parameters(), lr=1e-4)  # Adjust lr as needed

# Create DataLoader for training dataset
train_dataloader = DataLoader(
    train_dataset_tokenized,
    batch_size=training_args.per_device_train_batch_size,
    shuffle=True
)


## Description

* Set padding token.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# Ensure the tokenizer has a pad token for collator
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


## Description

* Create training DataLoader.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
from transformers import DataCollatorForLanguageModeling

# Use Hugging Face's data collator to ensure batches are tensors
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Re-create DataLoader with collate_fn
temp_train_dataloader = DataLoader(
    train_dataset_tokenized,
    batch_size=training_args.per_device_train_batch_size,
    shuffle=True,
    collate_fn=data_collator
)

# Optionally, replace the old train_dataloader
del train_dataloader
train_dataloader = temp_train_dataloader


## Description

* Set training epochs.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# Set the number of epochs for training
num_epochs = 2  # Changed from 1 to 2

## Description

* Create training DataLoader.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# --- Memory Optimization: Gradient Checkpointing, Mixed Precision, Batch Size, Accumulation, Cache Clearing ---
import torch
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader
from bitsandbytes.optim import AdamW8bit  # 8-bit optimizer for memory savings

# 1. Enable gradient checkpointing (for Hugging Face models)
try:
    moe_model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled.")
except Exception as e:
    print(f"Could not enable gradient checkpointing: {e}")

# 2. Reduce batch size (update your DataLoader or training_args)
if hasattr(training_args, 'per_device_train_batch_size'):
    training_args.per_device_train_batch_size = 1  # Set to 1 for minimal memory usage
    print(f"Batch size set to {training_args.per_device_train_batch_size}")

# Recreate DataLoader with batch_size=1 (DataLoader attributes are immutable after initialization)
train_dataloader = DataLoader(
    train_dataset_tokenized,
    batch_size=1,
    shuffle=True,
    collate_fn=data_collator
)
print(f"DataLoader recreated with batch_size={train_dataloader.batch_size}")

# 3. Gradient accumulation steps
accumulation_steps = 2  # Lowered for less memory pressure
# Use 8-bit AdamW optimizer for better memory efficiency
optimizer = AdamW8bit(moe_model.parameters(), lr=1e-4)

# 4. Free up GPU memory before training
import gc
gc.collect()
torch.cuda.empty_cache()

# 5. Example end-to-end training loop with AMP, device placement, accumulation, and cache clearing
for epoch in range(num_epochs):
    moe_model.train()
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        device = moe_model.device if hasattr(moe_model, 'device') else next(moe_model.parameters()).device


# 15.5.4 Evaluating the trained MoE baseline Model

## Description

* Define computational efficiency metrics.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# EVALUATE TRAINED MODEL - COMPLETE WITH ALL FIXES
print("="*70)
print("TRAINED MODEL EVALUATION")
print("="*70)

# ========== FIXED HELPER FUNCTIONS ==========

def compute_model_flops_fixed(model, seq_length=512):
    """Estimate FLOPs - handles PEFT models"""
    config = model.config
    n_layers = config.num_hidden_layers
    d_model = config.hidden_size
    intermediate_size = config.intermediate_size

    attention_flops = 4 * seq_length * d_model * d_model * n_layers
    ffn_flops = 8 * seq_length * d_model * intermediate_size * n_layers
    total_flops = attention_flops + ffn_flops

    # Try to detect MoE
    is_moe = False
    sparsity_factor = 1.0
    
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                num_experts_per_tok = layer.mlp.num_experts_per_tok
                sparsity_factor = num_experts_per_tok / num_experts
                break
    except:
        pass

    if is_moe:
        total_flops = attention_flops + (ffn_flops * sparsity_factor)

    return total_flops


def compute_parameter_efficiency_fixed(model, num_experts_per_tok=2):
    """Compute parameter efficiency - handles PEFT models"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Try to detect MoE
    is_moe = False
    active_params = total_params
    
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        # Count expert parameters across all MoE layers
        total_expert_params = 0
        num_experts = 0
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                
                # Sum expert FFN parameters
                for expert_idx in range(num_experts):
                    if hasattr(layer.mlp, 'gate_proj'):
                        total_expert_params += sum(p.numel() for p in layer.mlp.gate_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.up_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.down_proj[expert_idx].parameters())
        
        if is_moe and num_experts > 0:
            # Active expert params = (k/n) * total expert params
            sparsity = num_experts_per_tok / num_experts
            active_expert_params = int(total_expert_params * sparsity)
            
            # Non-expert params (attention, embeddings, etc)
            non_expert_params = total_params - total_expert_params
            
            # Total active = all non-expert params + active expert params
            active_params = non_expert_params + active_expert_params
        else:
            active_params = total_params
    except:
        pass
    
    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'active_params': active_params,
        'is_moe': is_moe
    }


# ========== EVALUATION ==========

# 1. MMLU Accuracy
print("\n Evaluating MMLU accuracy...")
trained_results = evaluate_mmlu_comprehensive(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,  
    show_progress=True
)
print(f"  Accuracy: {trained_results['accuracy']:.4f}")

# 2. FLOPs estimation
print("\n Computing FLOPs...")
trained_flops = compute_model_flops_fixed(moe_model, seq_length=MAX_LENGTH)

# 3. Throughput metrics
print("\n Measuring throughput...")
trained_throughput = compute_throughput_metrics(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

# 4. Parameter efficiency
print("\n Analyzing parameter efficiency...")
trained_params = compute_parameter_efficiency_fixed(
    model=moe_model,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK
)

# 5. Memory metrics
print("\n Collecting memory metrics...")
trained_memory = compute_memory_metrics(moe_model)

# Combine all metrics
trained_comprehensive = {
    **trained_results,
    'flops': trained_flops,
    **trained_throughput,
    **trained_params,
    **trained_memory,
}

print("\n Trained Model Evaluation Complete")
print(f"  Accuracy: {trained_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {trained_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {trained_comprehensive['ece']:.4f}")
print(f"  Throughput: {trained_comprehensive['throughput']:.2f} samples/sec")
print(f"  Avg Latency: {trained_comprehensive['avg_latency']:.4f} sec")

# COMPARISON: Before vs After Training
print("\n" + "="*70)
print("BEFORE vs AFTER TRAINING COMPARISON")
print("="*70)

import pandas as pd

comparison_df = pd.DataFrame({
    "Metric": [
        "MMLU Accuracy",
        "Top-2 Accuracy",
        "ECE (Calibration)",
        "Throughput (samples/sec)",
        "Avg Latency (sec)",
        "Correct Predictions",
        "Total Samples"
    ],
    "Before Training": [
        f"{moe_comprehensive['accuracy']:.4f}",
        f"{moe_comprehensive['top2_accuracy']:.4f}",
        f"{moe_comprehensive['ece']:.4f}",
        f"{moe_comprehensive['throughput']:.2f}",
        f"{moe_comprehensive['avg_latency']:.4f}",
        f"{moe_comprehensive['correct']}",
        f"{moe_comprehensive['total']}"
    ],
    "After Training": [
        f"{trained_comprehensive['accuracy']:.4f}",
        f"{trained_comprehensive['top2_accuracy']:.4f}",
        f"{trained_comprehensive['ece']:.4f}",
        f"{trained_comprehensive['throughput']:.2f}",
        f"{trained_comprehensive['avg_latency']:.4f}",
        f"{trained_comprehensive['correct']}",
        f"{trained_comprehensive['total']}"
    ],
    "Improvement": [
        f"{(trained_comprehensive['accuracy'] - moe_comprehensive['accuracy']) * 100:+.2f}%",
        f"{(trained_comprehensive['top2_accuracy'] - moe_comprehensive['top2_accuracy']) * 100:+.2f}%",
        f"{(trained_comprehensive['ece'] - moe_comprehensive['ece']):+.4f}",
        f"{(trained_comprehensive['throughput'] - moe_comprehensive['throughput']):+.2f}",
        f"{(trained_comprehensive['avg_latency'] - moe_comprehensive['avg_latency']):+.4f}",
        f"{(trained_comprehensive['correct'] - moe_comprehensive['correct']):+d}",
        "-"
    ]
})

print("\n", comparison_df.to_string(index=False))

# Save comparison to CSV
comparison_df.to_csv('training_comparison.csv', index=False)
print("\n Comparison saved to: training_comparison.csv")

# Save trained model metrics
trained_metrics_df = pd.DataFrame([trained_comprehensive])
trained_metrics_df.to_csv('trained_model_metrics.csv', index=False)
print("\n Trained metrics saved to: trained_model_metrics.csv")

## Description

* Calculate and display improvement metrics from training.
* This cell implements the functionality described above
* Results and outputs are displayed below


## Description

* Show improvement metrics.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# Calculate and display improvement between pre- and post-training metrics
if 'moe_comprehensive' in globals() and 'trained_comprehensive' in globals():
    print("\n==================== MoE Baseline Improvement ====================")
    for metric in ['accuracy', 'top2_accuracy', 'ece']:
        before = moe_comprehensive.get(metric, None)
        after = trained_comprehensive.get(metric, None)
        if before is not None and after is not None:
            diff = after - before
            print(f"{metric}: Before={before:.4f}, After={after:.4f}, Improvement={diff:+.4f}")
    print("=================================================================")

### 15.5.5 Router Statistics Analysis

Analyze router behavior to check for expert collapse and load balancing.

In [ ]:
# Collect router statistics from trained MoE model
print("Collecting router statistics from trained MoE model...")
print("This may take a few minutes...\n")
# Fix: Import defaultdict for router statistics collection
from collections import defaultdict

router_stats = collect_router_statistics(
    model=moe_model,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    answer_tokens=ANSWER_TOKENS,
    max_samples=500,
    device="cuda"
)

# Display key statistics
print("\n" + "="*70)
print("ROUTER STATISTICS SUMMARY")
print("="*70)

# Use correct key: 'expert_selections' with nested 'overall'
expert_usage = router_stats['expert_selections']['overall']
total_selections = sum(expert_usage)

print(f"\nTotal expert selections: {int(total_selections):,}")
print(f"\nExpert usage distribution:")
for expert_id in range(len(expert_usage)):
    count = expert_usage[expert_id]
    percentage = (count / total_selections) * 100 if total_selections > 0 else 0
    print(f"  Expert {expert_id}: {int(count):,} selections ({percentage:.2f}%)")

# Check for expert collapse
usage_percentages = [(count / total_selections) * 100 for count in expert_usage if total_selections > 0]
if usage_percentages:
    max_usage = max(usage_percentages)
    min_usage = min(usage_percentages)
    std_usage = np.std(usage_percentages)
    
    print(f"\nLoad balancing metrics:")
    print(f"  Max usage: {max_usage:.2f}%")
    print(f"  Min usage: {min_usage:.2f}%")
    print(f"  Std dev: {std_usage:.2f}%")
    print(f"  Expected (uniform): {100/len(expert_usage):.2f}%")
    
    if std_usage > 5.0:
        print("\n WARNING: High variance in expert usage detected!")
        print("  This suggests poor load balancing or expert collapse.")
    elif max_usage > 20.0:  # For 8 experts, expect ~12.5% each
        print("\n WARNING: Some experts are heavily overused!")
        print("  Router may be collapsing to a few experts.")
    else:
        print("\n Load balancing appears reasonable.")

# Calculate confidence metrics from the collected data
all_confidences = []
for layer_confs in router_stats['expert_confidence'].values():
    all_confidences.extend(layer_confs)

if all_confidences:
    avg_confidence = np.mean(all_confidences)
    max_confidence = np.max(all_confidences)
    print(f"\nAverage routing confidence: {avg_confidence:.4f}")
    print(f"Max routing confidence: {max_confidence:.4f}")


## Description

* Visualize router behavior.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# Visualize router statistics
visualize_router_statistics(router_stats, title="Trained MoE Router Analysis")

# 15.5.5 training with KD

## Description

* Load the language model with quantization.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# ============================================================================
# NEW CELL: Helper Functions for KD Training
# ============================================================================

def build_fresh_moe():
    """Build a fresh MoE model for KD training."""
    from transformers import AutoModelForCausalLM
    import gc
    import torch
    
    # Load fresh base model
    fresh_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    # Apply MoE transformation
    fresh_moe = replace_ffn_with_moe(
        model=fresh_model,
        num_experts=NUM_EXPERTS,
        num_experts_per_tok=NUM_EXPERTS_PER_TOK,
        router_jitter_noise=ROUTER_JITTER_NOISE,
        router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
        bnb_config=bnb_config,
        ram_threshold=80.0,
        use_disk_offload=True
    )
    
    torch.cuda.empty_cache()
    gc.collect()
    
    return fresh_moe


def apply_lora_to_model(model):
    """Apply LoRA configuration to a model."""
    from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
    
    # Prepare for training
    model = prepare_model_for_kbit_training(model)
    
    # LoRA config
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        modules_to_save=["router"],
        use_rslora=False,
    )
    
    model = get_peft_model(model, lora_config)
    model.train()
    
    return model

print(" Helper functions defined for KD training")

## Description

* Configure training arguments.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:

# ============================================================================
# EXAMPLE: HUGGING FACE TRAINER (RECOMMENDED)
# ============================================================================
# This is the RECOMMENDED way to train - handles device placement automatically

print("Setting up Hugging Face Trainer...")

# Training arguments
training_args = TrainingArguments(
    output_dir=TRAINING_CONFIG["output_dir"],
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_steps=TRAINING_CONFIG["warmup_steps"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    fp16=False,
    bf16=True,  # Match model dtype
    gradient_checkpointing=False,  # Enable if OOM
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
)

# Create trainer
trainer = MoETrainer(
    model=moe_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    router_aux_loss_coef=TRAINING_CONFIG["router_aux_loss_coef"],
)

print(" Trainer initialized")
print("\nTo train, run: trainer.train()")
print("\n" + "="*70)

# ============================================================================
# ALTERNATIVE: MANUAL TRAINING LOOP (if needed)
# ============================================================================
# Only use this if you need custom training logic
# WARNING: You MUST handle device placement manually!

def manual_training_loop(model, train_dataset, num_epochs=1, batch_size=1):
    """
    Manual training loop with proper device handling.
    NOTE: HuggingFace Trainer is recommended - use this only if absolutely necessary.
    """
    from torch.utils.data import DataLoader
    import torch
    
    # Setup device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Ensure model is on GPU
    model = model.to(device)
    model.train()
    
    # Create dataloader
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator,  # Use the data collator
    )
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    
    print(f"\nStarting manual training loop...")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Total batches: {len(train_dataloader)}")
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        epoch_loss = 0
        
        for step, batch in enumerate(train_dataloader):
            # CRITICAL: Move batch to GPU
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            
            # Backward pass
            loss.backward()
            
            # Check gradients are flowing
            if step == 0:
                grad_norm = sum(p.grad.abs().sum().item() for p in model.parameters() if p.grad is not None)
                print(f"  Step {step}: loss={loss.item():.4f}, grad_norm={grad_norm:.2e}")
                if grad_norm == 0:
                    print("  WARNING: Gradients are zero! Training will not work!")
            
            optimizer.step()
            
            epoch_loss += loss.item()
            
            if step % 10 == 0:
                print(f"  Step {step}: loss={loss.item():.4f}")
        
        avg_loss = epoch_loss / len(train_dataloader)
        print(f"  Epoch {epoch + 1} avg loss: {avg_loss:.4f}")
    
    print("\nTraining complete!")
    return model

# Example usage (commented out - use Trainer instead):
# moe_model = manual_training_loop(moe_model, train_dataset, num_epochs=1, batch_size=1)


### 15.6 MoE Variant Experiment System

System for testing different MoE configurations:
1. **Routing Strategies**: Top-1, Top-2, Top-K routing
2. **Expert Counts**: 4, 8, 16 experts
3. **Expert Placement**: All layers, every 2nd layer, selected layers
4. **Specialized Configurations**: Different intermediate sizes

## How to Improve MoE Baseline Accuracy

- **Warm-start experts:** Initialize MoE experts with weights from a pretrained dense model instead of random weights.
- **Router initialization:** If possible, initialize the router to distribute tokens more evenly (not purely random).
- **Increase active experts:** Temporarily increase the number of active experts per token for the baseline.
- **Check implementation:** Ensure routing, masking, and expert selection logic is correct.
- **Use load balancing loss:** Add a load balancing loss to encourage even expert utilization.
- **Monitor expert utilization:** Visualize expert usage and router confidence to debug and improve routing.

These steps will help reduce the steep decline in accuracy when creating the baseline MoE model.

In [ ]:
# Ensure all MoE parameters are trainable (experts and router)
for name, param in moe_model.named_parameters():
    if not param.requires_grad:
        param.requires_grad = True
        print(f"Set requires_grad=True for {name}")

# Create optimizer using only trainable parameters
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, moe_model.parameters()), lr=1e-4)
print("Optimizer created with trainable parameters only.")

### Optional: Run MoE Variant Experiments

**Note:** This section is optional and will take significant time (2-4 hours for all variants).
Comment out if you want to skip variant experiments and just use the baseline MoE.

In [ ]:
import wandb
import traceback
import gc
import torch

print(" RETRYING EXPERIMENTS WITH CPU QUANTIZATION FIX")

RUN_VARIANT_EXPERIMENTS = True  # Set to True to run experiments

# Ensure wandb is initialized
try:
    if wandb.run is None:
        wandb.init(project="mistral-moe-variants", name="variant-experiments")
except:
    pass

if RUN_VARIANT_EXPERIMENTS:
    print("RUNNING MOE VARIANT EXPERIMENTS")

    # Force cleanup before starting
    torch.cuda.empty_cache()
    gc.collect()

    # Create experiment runner
    # Pass base_model=None since the runner loads its own fresh model internally
    runner = MoEExperimentRunner(
        base_model=None,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        answer_tokens=ANSWER_TOKENS
    )

    # Select experiments to run
    experiments_to_run = [
        "efficient_4x1",  # Faster variant
        "baseline_8x2",  # Standard variant
        "top1_8x1",  # Different routing
        "large_16x2",  # Uncomment for large variant (slower)
        "sparse_8x2",  # Uncomment for sparse placement
        "placement_early_8x2",
        "placement_middle_8x2",
        "placement_late_8x2",
        "routing_noisy_8x2",
        "moe_baseline",
        "balanced_8x2",
        "placement_mixed_8x2",
    ]

    # Run each experiment
    for exp_name in experiments_to_run:
        if exp_name in EXPERIMENT_CONFIGS:
            config = EXPERIMENT_CONFIGS[exp_name]

            try:
                results = runner.run_experiment(config, max_samples=1000)  # Changed from 5000 to 1000

                # Add to dashboard if available
                if 'dashboard' in globals():
                    dashboard.add_experiment(exp_name, results)

                # Log to wandb
                wandb.log({
                    f'variants/{exp_name}/accuracy': results['accuracy'],
                    f'variants/{exp_name}/tokens_per_second': results['tokens_per_second'],
                    f'variants/{exp_name}/flops': results['flops']/1e9,
                })

            except Exception as e:
                print(f" Error in experiment {exp_name}: {str(e)}")
                traceback.print_exc()
                continue

    # Generate comparison
    print("VARIANT EXPERIMENT COMPARISON")
    runner.compare_experiments()

    print("\n Variant experiments complete!")

In [ ]:
# Display the comprehensive comparison table from the dashboard
if 'dashboard' in globals() and dashboard.experiments:
    print("\n" + "="*100)
    print("MOE EXPERIMENT COMPARISON DASHBOARD")
    print("="*100 + "\n")

    # Generate and display the table
    comp_df = dashboard.generate_comparison_table()
    display(comp_df)

    # Optional: Highlight best metrics
    if not comp_df.empty:
        best_acc = comp_df.loc[comp_df['Accuracy'].astype(float).idxmax()]
        print(f"\n Best Accuracy: {best_acc['Experiment']} ({best_acc['Accuracy']})")
    else:
        print("Dashboard not populated yet. Please run the experiments/evaluations first.")

## Description

* Define smart variant selector.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
import numpy as np
from scipy.stats import pearsonr

class SmartMoESelector:
    """
    Implements the 'Pre-Training Prediction' workflow to efficiently select
    and train only the most promising MoE variants.
    """

    def __init__(self, runner):
        self.runner = runner
        self.predictions = {}
        self.outcomes = {}
        self.rankings = []

    def run_prediction_phase(self, configs, max_samples=1000):
        """
        Phase 1: Evaluate all variants in frozen/sparse-upcycled state.
        """
        print("\n" + "="*80)
        print("PHASE 1: PREDICTION (Evaluating Frozen Variants)")
        print("="*80)

        for name, config in configs.items():
            print(f"\n Assessing variant: {name}...")
            # Run with train=False to get pre-training metrics
            results = self.runner.run_experiment(config, max_samples=max_samples, train=False)

            # Calculate Load Balance Std (lower is better)
            # Use router stats if available, else heuristic (0 if missing to avoid breaking)
            lb_std = 0.0
            if 'router_stats' in results and 'expert_selections' in results['router_stats']:
                counts = results['router_stats']['expert_selections']['overall']
                if counts.sum() > 0:
                    probs = counts / counts.sum()
                    lb_std = np.std(probs) * 100  # percentage

            # Calculate Efficiency Score (FLOPs / Throughput) (lower is better)
            # E.g. 1e12 flops / 100 tok/s = 1e10 cost per token speed
            flops = results.get('flops', 1e12)
            throughput = results.get('tokens_per_second', 1.0)
            efficiency_score = flops / throughput if throughput > 0 else 1e12

            self.predictions[name] = {
                'accuracy': results['accuracy'],
                'load_balance_std': lb_std,
                'efficiency_score': efficiency_score,
                'full_results': results
            }

    def rank_variants(self):
        """
        Rank variants based on weighted criteria:
        1. Accuracy (50%) - Higher is better
        2. Load Balance (30%) - Lower std dev is better
        3. Efficiency (20%) - Lower cost/speed ratio is better
        """
        print("\n" + "="*80)
        print("RANKING CANDIDATES")
        print("="*80)

        # Normalize metrics for scoring
        accs = [m['accuracy'] for m in self.predictions.values()]
        lbs = [m['load_balance_std'] for m in self.predictions.values()]
        effs = [m['efficiency_score'] for m in self.predictions.values()]

        max_acc = max(accs) if accs else 1.0
        max_lb = max(lbs) if lbs and max(lbs) > 0 else 1.0
        min_eff = min(effs) if effs else 1.0

        # Create ranked list
        ranking_data = []
        for name, m in self.predictions.items():
            # Score calculation (Normalize to 0-1 range approx)
            # Acc: Higher is better
            s_acc = m['accuracy'] / max_acc if max_acc > 0 else 0

            # LB: Lower is better (invert)
            s_lb = 1.0 - (m['load_balance_std'] / max_lb) if max_lb > 0 else 0

            # Eff: Lower is better (invert)
            s_eff = min_eff / m['efficiency_score'] if m['efficiency_score'] > 0 else 0

            # Weighted Sum: 50% Acc + 30% LB + 20% Eff
            total_score = (s_acc * 0.5) + (s_lb * 0.3) + (s_eff * 0.2)

            ranking_data.append({
                'name': name,
                'score': total_score,
                'accuracy': m['accuracy'],
                'load_balance_std': m['load_balance_std'],
                'efficiency': m['efficiency_score']
            })

        # Sort by score descending
        self.rankings = sorted(ranking_data, key=lambda x: x['score'], reverse=True)

        # Display Rankings
        print(f"\n{'Rank':<5} {'Name':<25} {'Score':<8} {'Acc':<8} {'Bal(%)':<8} {'Eff':<10}")
        print("-" * 70)
        for i, r in enumerate(self.rankings, 1):
            print(f"{i:<5} {r['name']:<25} {r['score']:.4f} {r['accuracy']:.4f} {r['load_balance_std']:<8.2f} {r['efficiency']:.2e}")

        return self.rankings

    def train_top_candidates(self, configs, top_n=2, train_steps=50):
        """
        Phase 2: Train only the top N candidates.
        """
        print("\n" + "="*80)
        print(f"PHASE 2: TRAINING TOP {top_n} CANDIDATES")
        print("="*80)

        top_names = [r['name'] for r in self.rankings[:top_n]]

        for name in top_names:
            print(f"\n Training selected candidate: {name}...")
            config = configs[name]

            # Run with train=True
            # Using same max_samples for post-eval to keep comparison fair
            results = self.runner.run_experiment(config, max_samples=500, train=True, train_steps=train_steps)

            self.outcomes[name] = results

            # Calculate gain
            pre_acc = self.predictions[name]['accuracy']
            post_acc = results['accuracy']
            print(f" Gain: {post_acc - pre_acc:+.4f} ({pre_acc:.4f} -> {post_acc:.4f})")

    def validate_predictions(self):
        """
        Phase 3: Validate if pre-training metrics correlated with post-training success.
        """
        print("\n" + "="*80)
        print("PHASE 3: VALIDATION")
        print("="*80)

        trained_names = list(self.outcomes.keys())
        if len(trained_names) < 2:
            print(" Need at least 2 trained models to calculate correlation.")
            return

        pre_accs = [self.predictions[n]['accuracy'] for n in trained_names]
        post_accs = [self.outcomes[n]['accuracy'] for n in trained_names]

        correlation, _ = pearsonr(pre_accs, post_accs)

        print(f" Trained Variants: {trained_names}")
        print(f" Pre-Train Accuracies: {pre_accs}")
        print(f" Post-Train Accuracies: {post_accs}")
        print(f"\n Pearson Correlation: {correlation:.4f}")

        if correlation > 0.8:
            print(" STRONG correlation! Prediction strategy is highly effective.")
        elif correlation > 0.5:
            print(" MODERATE correlation. Strategy is useful but not perfect.")
        else:
            print(" WEAK correlation. Selection criteria may need adjustment.")


# Helper function to run the whole campaign
def run_smart_moe_campaign(runner, configs, top_n=2, train_steps=50):
    selector = SmartMoESelector(runner)

    # 1. Predict
    selector.run_prediction_phase(configs, max_samples=500)

    # 2. Rank
    selector.rank_variants()

    # 3. Train Top Candidates
    selector.train_top_candidates(configs, top_n=top_n, train_steps=train_steps)

    # 4. Validate
    selector.validate_predictions()

    return selector

print(" Smart MoE Selector defined")

## Description

* Execute smart selection campaign to identify best MoE variants.
* This cell implements the functionality described above
* Results and outputs are displayed below


## Description

* Run variant selection.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# EXECUTE SMART MOE CAMPAIGN

print(" STARTING SMART MOE CAMPAIGN")
print("="*80)

# Run the full workflow
# 1. Evaluate all variants (Fast mode: n=500) to get prediction metrics
# 2. Rank and select top 2
# 3. Train top 2 (Fast train: 50 steps)
# 4. Validate results

selector = run_smart_moe_campaign(
    runner=MoEExperimentRunner(
        base_model=None,  # Runner creates its own models
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        answer_tokens=ANSWER_TOKENS,
        train_dataset=train_dataset_tokenized
    ),
    configs=EXPERIMENT_CONFIGS,
    top_n=11,  # Select top 2 variants for training
    train_steps=50  # Short training for demonstration (increase for real results)
)

print("\n CAMPAIGN COMPLETE")

## Description

* Collect router statistics.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
print("MOE ROUTER ANALYSIS")


# Collect router statistics
router_stats = collect_router_statistics(
    model=moe_model,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    answer_tokens=ANSWER_TOKENS,
    max_samples=500,  # Use subset for faster analysis
    device="cuda"
)

# Visualize router behavior
visualize_router_statistics(
    router_stats=router_stats,
    title="MoE Router Analysis: Baseline Model"
)

print("\n Router analysis complete!")

## Description

* Run baseline model evaluation.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# Save MoE evaluation metrics
moe_metrics = {
    'accuracy': moe_comprehensive['accuracy'],
    'top2_accuracy': moe_comprehensive['top2_accuracy'],
    'ece': moe_comprehensive['ece'],
    'throughput': moe_comprehensive['throughput'],
    'avg_latency': moe_comprehensive['avg_latency'],
    'correct': moe_comprehensive['correct'],
    'total': moe_comprehensive['total'],
}

moe_metrics_df = pd.DataFrame([moe_metrics])
moe_metrics_df.to_csv('moe_baseline_metrics.csv', index=False)
print("MoE baseline metrics saved to moe_baseline_metrics.csv")

# Create confusion matrix heatmap for MoE
plt.figure(figsize=(8, 6))
sns.heatmap(
    moe_comprehensive['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['A', 'B', 'C', 'D'],
    yticklabels=['A', 'B', 'C', 'D']
)
plt.title('MoE Baseline Model Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('moe_confusion_matrix.png', dpi=150)

# Log confusion matrix to WandB safely
try:
    if wandb.run is not None:
        wandb.log({"moe_confusion_matrix": wandb.Image('moe_confusion_matrix.png')})
except Exception as e:
    print(f"WandB logging skipped: {e}")

# Log confusion matrix to TensorBoard
try:
    from PIL import Image
    import numpy as np
    moe_confusion_img = Image.open('moe_confusion_matrix.png')
    moe_confusion_array = np.array(moe_confusion_img)
    if 'tb_writer' in globals() and tb_writer is not None:
        tb_writer.add_image('moe_confusion_matrix', moe_confusion_array, 0, dataformats='HWC')
    else:
        print("TensorBoard writer not initialized, skipping image logging.")
except Exception as e:
    print(f"TensorBoard logging skipped: {e}")

print("MoE confusion matrix saved to moe_confusion_matrix.png")

print("\nMoE Baseline Evaluation Summary:")
print(f"  Accuracy:      {moe_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {moe_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE:           {moe_comprehensive['ece']:.4f}")
print(f"  Throughput:    {moe_comprehensive['throughput']:.2f} samples/sec")
print(f"  Avg Latency:   {moe_comprehensive['avg_latency']:.4f} sec/sample")


## Description

* Apply LoRA adapters.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# APPLY LORA TO EXPERTS + ATTENTION + ROUTER

from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training, PeftModel
import torch
import gc
from bitsandbytes.nn import Linear4bit  # OPTION 2: Changed from Linear8bitLt

print("="*70)
print("APPLYING LORA TO EXPERTS + ATTENTION + ROUTER")
print("="*70)

# Check if moe_model is already a PeftModel and unload if necessary
if isinstance(moe_model, PeftModel):
    print("Warning: moe_model is already a PeftModel. Unloading to re-apply LoRA.")
    moe_model = moe_model.unload()
    torch.cuda.empty_cache()
    gc.collect()

# CRITICAL: Prepare quantized model for training
moe_model = prepare_model_for_kbit_training(moe_model)

# Configure LoRA for BOTH attention AND MoE experts
lora_config = LoraConfig(
    r=16,  # Low-rank dimension
    lora_alpha=32,  # Scaling factor
    target_modules=[
        Linear4bit  # OPTION 2: Changed from Linear8bitLt
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    modules_to_save=["router"]  # Also train router weights directly (named 'router' in MoELayer)
)

# Apply LoRA
moe_model = get_peft_model(moe_model, lora_config)

print("\n LoRA Configuration:")
print(f" Rank: {lora_config.r}")
print(f" Alpha: {lora_config.lora_alpha}")
print(f" Target modules: {lora_config.target_modules}")
print(f" Modules to save (routers): {lora_config.modules_to_save}")

print("\n Trainable Parameters:")
moe_model.print_trainable_parameters()

# Expected output:
# trainable params: ~420M || all params: ~7.4B || trainable%: 5.67%

# Ensure training mode
moe_model.train()

print("\n" + "="*70)
print(" EXPERT TRAINING ENABLED!")
print("="*70)
print("\n What's trainable:")
print(" Attention LoRA adapters: ~16.8M params")
print(" Expert LoRA adapters: ~400M params (NEW!)")
print(" Router gates: ~1M params")
print("\n What's frozen:")
print(" Quantized base weights (4-bit storage)")
print("\n This enables expert specialization while maintaining memory efficiency!\n")

## Description

* Configure training arguments.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# SETUP TRAINING ARGUMENTS FOR EXPERT TRAINING

from transformers import TrainingArguments

OUTPUT_DIR = "./mistral_moe_expert_trained"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=1000,
    num_train_epochs=1,  # Start with 1 epoch

    # REDUCED batch size due to expert training
    per_device_train_batch_size=1,  # Reduced from 4
    gradient_accumulation_steps=8,  # Increased from 2
    # Effective batch = 1 × 8 = 8 samples

    # Learning rate for expert training
    learning_rate=2e-4,  # Lower LR for experts (vs 5e-4)
    weight_decay=0.01,

    # Evaluation
    logging_steps=25,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,

    # Memory optimizations
    bf16=True,
    gradient_checkpointing=True,  # CRITICAL for expert training
    dataloader_num_workers=2,  # Reduced from 4
    dataloader_pin_memory=True,

    # Optimizer
    optim="paged_adamw_8bit",

    # Warmup
    warmup_ratio=0.03,  # 3% warmup

    # Save best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Experiment tracking
    report_to="wandb",
    disable_tqdm=False,
    save_total_limit=2,
    seed=42,
)

print("="*70)
print("TRAINING ARGUMENTS FOR EXPERT TRAINING")
print("="*70)
print(f"\n Batch Configuration:")
print(f" Per-device batch size: {training_args.per_device_train_batch_size}")
print(f" Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f" Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

print(f"\n Training Configuration:")
print(f" Epochs: {training_args.num_train_epochs}")
print(f" Learning rate: {training_args.learning_rate}")
print(f" Optimizer: {training_args.optim}")
print(f" Precision: bfloat16")
print(f" Gradient checkpointing: {training_args.gradient_checkpointing}")

print(f"\n Output:")
print(f" Directory: {OUTPUT_DIR}")

print(f"\n⏱ Expected training time:")
print(f" ~6-8 hours for 1 epoch (with expert training)")

print("\n Memory Requirements:")
print(" GPU VRAM: ~14-18GB (with gradient checkpointing)")
print(" System RAM: ~32GB recommended")
print("="*70)

### 15.7 Save MoE Configuration

Save MoE model configuration and evaluation settings for reference.


In [ ]:
# Save MoE evaluation configuration for reference
import json

moe_config_dict = {
    'base_model': model_id,
    'architecture': 'MoE (Mixture of Experts)',
    'num_experts': NUM_EXPERTS,
    'num_experts_per_tok': NUM_EXPERTS_PER_TOK,
    'router_jitter_noise': ROUTER_JITTER_NOISE,
    'router_aux_loss_coef': ROUTER_AUX_LOSS_COEF,
    'dataset': 'MMLU',
    'eval_samples': len(eval_dataset),
    'quantization': '4-bit (NF4)',
    'sparse_upcycling': True,
    'moe_accuracy': float(moe_results['accuracy']),
    'moe_top2_accuracy': float(moe_results['top2_accuracy']),
    'moe_ece': float(moe_results['ece']),
    'moe_throughput': float(moe_results['throughput']),
    'moe_avg_latency': float(moe_results['avg_latency']),
}

with open('moe_baseline_config.json', 'w') as f:
    json.dump(moe_config_dict, f, indent=2)

print("MoE baseline evaluation config saved to moe_baseline_config.json")


## Performance Summary

All models evaluated with consistent 0-shot prompting.

### 15.8 Generate MoE Baseline Report

Generate a comprehensive markdown report summarizing the MoE baseline evaluation.


In [ ]:
moe_report = f"""
# Mistral-7B MoE Baseline (Frozen) Evaluation Report

## Experiment Overview

**Model:** {model_id} (MoE Architecture)
**Architecture:** Sparse Mixture of Experts ({NUM_EXPERTS} experts, top-{NUM_EXPERTS_PER_TOK} routing)
**Dataset:** MMLU (Massive Multitask Language Understanding)
**Method:** MoE baseline evaluation with 4-bit quantization and sparse upcycling (frozen experts)
**Evaluation Samples:** {moe_comprehensive.get('total', 0):,}

## Model Configuration

- **Architecture:** MoE ({NUM_EXPERTS} experts per layer, top-{NUM_EXPERTS_PER_TOK} routing)
- **Quantization:** 4-bit NF4
- **Precision:** BFloat16
- **Max Sequence Length:** {MAX_LENGTH}
- **Router Jitter Noise:** {ROUTER_JITTER_NOISE}
- **Auxiliary Loss Coefficient:** {ROUTER_AUX_LOSS_COEF}
- **Sparse Upcycling:** Enabled (FFN weights duplicated to all experts)

## Results Summary

### Priority 1 Metrics (Core Performance)

| Metric | Value |
|--------|-------|
| MMLU Accuracy | {moe_comprehensive.get('accuracy', 0):.4f} |

### Priority 2 Metrics (Quality & Efficiency)

| Metric | Value |
|--------|-------|
| Throughput (samples/sec) | {moe_comprehensive.get('samples_per_second', 0):.2f} |
| Throughput (tokens/sec)  | {moe_comprehensive.get('tokens_per_second', 0):.2f} |
| Avg Latency (ms/token)   | {moe_comprehensive.get('ms_per_token', 0):.2f} |
| ECE (Calibration) | {moe_comprehensive.get('ece', 0):.4f} |

### Priority 3 Metrics (Additional Insights)

| Metric | Value |
|--------|-------|
| Top-2 Accuracy | {moe_comprehensive.get('top2_accuracy', 0):.4f} |
| FLOPs per forward (G) | {moe_comprehensive.get('flops', 0)/1e9:.2f} |
| Total Parameters (B) | {moe_comprehensive.get('total_params', 0)/1e9:.2f} |
| Active Parameters (B) | {moe_comprehensive.get('active_params', 0)/1e9:.2f} |
| Trainable Parameters (M) | {moe_comprehensive.get('trainable_params', 0)/1e6:.2f} |
| Sparsity Ratio | {moe_comprehensive.get('sparsity_ratio', 0):.2%} |
| Model Size (MB) | {moe_comprehensive.get('model_size_mb', 0):.2f} |
| GPU Allocated (GB) | {moe_comprehensive.get('gpu_memory_allocated_gb', 0):.2f} |
| GPU Reserved (GB) | {moe_comprehensive.get('gpu_memory_reserved_gb', 0):.2f} |

## Confusion Matrix

```
     A    B    C    D
A  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[0][0]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[0][1]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[0][2]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[0][3]:3d}
B  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[1][0]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[1][1]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[1][2]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[1][3]:3d}
C  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[2][0]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[2][1]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[2][2]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[2][3]:3d}
D  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[3][0]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[3][1]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[3][2]:3d}  {moe_comprehensive.get('confusion_matrix', [[0]*4]*4)[3][3]:3d}
```

## Key Findings

1.  **Performance:** The MoE baseline (frozen) model achieved {moe_comprehensive.get('accuracy', 0):.2%} accuracy on MMLU.

2.  **Calibration:** ECE of {moe_comprehensive.get('ece', 0):.4f} indicates {'good' if moe_comprehensive.get('ece', 1) < 0.1 else 'moderate'} calibration.

3.  **Top-2 Performance:** {moe_comprehensive.get('top2_accuracy', 0):.2%} top-2 accuracy shows the model frequently includes the correct answer in its top 2 choices.

4.  **Efficiency:** The model processes {moe_comprehensive.get('samples_per_second', 0):.2f} samples per second with an average latency of {moe_comprehensive.get('avg_latency', 0):.4f} seconds per sample.

5.  **Architecture:** The MoE model uses {NUM_EXPERTS} experts per layer with top-{NUM_EXPERTS_PER_TOK} routing, providing approximately {moe_comprehensive.get('total_params', 0)/1e9:.2f}B total parameters with ~{moe_comprehensive.get('active_params', 0)/1e9:.2f}B active parameters per forward pass.

## Files Generated

- `moe_baseline_metrics.csv`: Evaluation metrics
- `moe_baseline_config.json`: Model and evaluation configuration
- `moe_confusion_matrix.png`: Confusion matrix heatmap

## Experiment Tracking

All metrics were logged to:
- **Weights & Biases:** Project `mistral-mmlu-baseline`
- **TensorBoard:** `./logs/tensorboard` directory

---

*Report generated automatically at the end of MoE baseline evaluation*
"""

# Save report
with open('moe_baseline_report.md', 'w') as f:
    f.write(moe_report)

print("\n MoE baseline report saved to moe_baseline_report.md")


## Description

* Define results dashboard.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

class ExperimentDashboard:
    """Dashboard for comparing all experiments."""

    def __init__(self):
        self.experiments = {}

    def add_experiment(self, name, results):
        """Add experiment results to dashboard."""
        self.experiments[name] = results

    def generate_comparison_table(self):
        """Generate comprehensive comparison table."""
        data = []

        for name, results in self.experiments.items():
            row = {
                'Experiment': name,
                'Accuracy': f"{results.get('accuracy', 0):.4f}",
                'Top-2': f"{results.get('top2_accuracy', 0):.4f}",
                'ECE': f"{results.get('ece', 0):.4f}",
                'Tokens/sec': f"{results.get('tokens_per_second', 0):.2f}",
                'FLOPs (G)': f"{results.get('flops', 0) / 1e9:.2f}",
                'Active Params (B)': f"{results.get('active_params', 0) / 1e9:.2f}",
                'GPU Memory (GB)': f"{results.get('gpu_memory_allocated_gb', 0):.2f}",
                'Sparsity': f"{results.get('sparsity_ratio', 1.0):.2%}",
            }
            data.append(row)

        df = pd.DataFrame(data)
        return df

    def plot_efficiency_comparison(self, save_path='results/efficiency_comparison.png'):
        """Plot comprehensive efficiency comparisons."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        # Prepare data
        names = list(self.experiments.keys())
        accuracies = [self.experiments[n].get('accuracy', 0) * 100 for n in names]
        tokens_per_sec = [self.experiments[n].get('tokens_per_second', 0) for n in names]
        flops = [self.experiments[n].get('flops', 0)/1e9 for n in names]
        memory = [self.experiments[n].get('gpu_memory_allocated_gb', 0) for n in names]
        active_params = [self.experiments[n].get('active_params', 0)/1e9 for n in names]

        # Plot 1: Accuracy vs Throughput
        axes[0, 0].scatter(tokens_per_sec, accuracies, s=150, alpha=0.7, c=range(len(names)), cmap='viridis')
        for i, name in enumerate(names):
            axes[0, 0].annotate(name, (tokens_per_sec[i], accuracies[i]),
                                fontsize=8, xytext=(5, 5), textcoords='offset points')
        axes[0, 0].set_xlabel('Throughput (tokens/sec)', fontsize=11)
        axes[0, 0].set_ylabel('Accuracy (%)', fontsize=11)
        axes[0, 0].set_title('Accuracy vs Throughput Trade-off', fontsize=12, fontweight='bold')
        axes[0, 0].grid(alpha=0.3)

        # Plot 2: Accuracy vs FLOPs
        axes[0, 1].scatter(flops, accuracies, s=150, alpha=0.7, c=range(len(names)), cmap='plasma')
        for i, name in enumerate(names):
            axes[0, 1].annotate(name, (flops[i], accuracies[i]),
                                fontsize=8, xytext=(5, 5), textcoords='offset points')
        axes[0, 1].set_xlabel('FLOPs (Billions)', fontsize=11)
        axes[0, 1].set_ylabel('Accuracy (%)', fontsize=11)
        axes[0, 1].set_title('Accuracy vs Computational Cost', fontsize=12, fontweight='bold')
        axes[0, 1].grid(alpha=0.3)

        # Plot 3: Accuracy vs Memory
        axes[1, 0].scatter(memory, accuracies, s=150, alpha=0.7, c=range(len(names)), cmap='coolwarm')
        for i, name in enumerate(names):
            axes[1, 0].annotate(name, (memory[i], accuracies[i]),
                                fontsize=8, xytext=(5, 5), textcoords='offset points')
        axes[1, 0].set_xlabel('GPU Memory (GB)', fontsize=11)
        axes[1, 0].set_ylabel('Accuracy (%)', fontsize=11)
        axes[1, 0].set_title('Accuracy vs Memory Usage', fontsize=12, fontweight='bold')
        axes[1, 0].grid(alpha=0.3)

        # Plot 4: Accuracy comparison bars
        colors = plt.cm.viridis(np.linspace(0, 1, len(names)))
        bars = axes[1, 1].barh(names, accuracies, color=colors, alpha=0.7)
        axes[1, 1].set_xlabel('Accuracy (%)', fontsize=11)
        axes[1, 1].set_title('Accuracy Comparison', fontsize=12, fontweight='bold')
        axes[1, 1].grid(axis='x', alpha=0.3)

        # Add value labels on bars
        for bar in bars:
            width = bar.get_width()
            axes[1, 1].text(width, bar.get_y() + bar.get_height()/2,
                            f'{width:.2f}%', ha='left', va='center', fontsize=9)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f" Visualization saved to {save_path}")
        plt.show()

        return fig

    def generate_report(self, save_path='results/experiment_report.md'):
        """Generate comprehensive markdown report."""
        report = "# MoE Experiment Comprehensive Report\n\n"
        report += f"**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n"
        report += f"**Total Experiments:** {len(self.experiments)}\n\n"

        # Add comparison table
        df = self.generate_comparison_table()
        report += "## Experiment Comparison Table\n\n"
        report += df.to_markdown(index=False)
        report += "\n\n"

        # Find best performers
        report += "## Best Performers\n\n"

        accuracies = {name: results.get('accuracy', 0) for name, results in self.experiments.items()}
        throughputs = {name: results.get('tokens_per_second', 0) for name, results in self.experiments.items()}

        best_accuracy = max(accuracies.items(), key=lambda x: x[1])
        best_throughput = max(throughputs.items(), key=lambda x: x[1])

        report += f"### Highest Accuracy\n"
        report += f"- **{best_accuracy[0]}**: {best_accuracy[1]:.4f}\n\n"

        report += f"### Highest Throughput\n"
        report += f"- **{best_throughput[0]}**: {best_throughput[1]:.2f} tokens/sec\n\n"

        # Detailed sections
        report += "## Detailed Results by Experiment\n\n"
        for name, results in self.experiments.items():
            report += f"### {name}\n\n"
            report += f"- **Accuracy:** {results.get('accuracy', 0):.4f}\n"
            report += f"- **Top-2 Accuracy:** {results.get('top2_accuracy', 0):.4f}\n"
            report += f"- **ECE:** {results.get('ece', 0):.4f}\n"
            report += f"- **Throughput:** {results.get('tokens_per_second', 0):.2f} tokens/sec\n"
            report += f"- **FLOPs:** {results.get('flops', 0)/1e9:.2f}G\n"
            report += f"- **Active Params:** {results.get('active_params', 0)/1e9:.2f}B\n"
            report += f"- **GPU Memory:** {results.get('gpu_memory_allocated_gb', 0):.2f} GB\n"
            report += "\n"

        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, 'w') as f:
            f.write(report)

        print(f" Report saved to {save_path}")
        return report

## Description

* Populate results dashboard.
* This cell implements the functionality described above
* Results and outputs are displayed below


In [ ]:
# Populate dashboard with all available results
print("\n" + "="*80)
print("POPULATING EXPERIMENT DASHBOARD")
print("="*80 + "\n")

# Add baseline if exists
if 'baseline_comprehensive' in globals():
    dashboard.add_experiment('Dense Baseline', baseline_comprehensive)
    print(" Added: Dense Baseline")

# Add MoE baseline if exists
if 'moe_comprehensive' in globals():
    dashboard.add_experiment('MoE 8x2 Frozen', moe_comprehensive)
    print(" Added: MoE 8x2 Frozen")

# Add trained MoE if exists
if 'final_comprehensive' in globals():
    dashboard.add_experiment('MoE 8x2 Trained', final_comprehensive)
    print(" Added: MoE 8x2 Trained")

print(f"\n Total experiments in dashboard: {len(dashboard.experiments)}")

# Generate comparison table
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON TABLE")
print("="*80 + "\n")

comparison_df = dashboard.generate_comparison_table()
print(comparison_df.to_string(index=False))

# Generate visualizations
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80 + "\n")

if len(dashboard.experiments) >= 2:
    dashboard.plot_efficiency_comparison()
else:
    print(" Need at least 2 experiments for comparison plots")

# Generate markdown report
print("\n" + "="*80)
print("GENERATING COMPREHENSIVE REPORT")
print("="*80 + "\n")

report = dashboard.generate_report()

# Export comparison to CSV
csv_path = 'results/comprehensive_comparison.csv'
comparison_df.to_csv(csv_path, index=False)
print(f" Comparison CSV saved to {csv_path}")

print("\n" + "="*80)
print("EXPERIMENT DASHBOARD COMPLETE")
print("="*80)

### 15.9 Display MoE Metrics Summary

Show a formatted summary of MoE baseline evaluation metrics.


In [ ]:
print("MOE BASELINE METRICS SUMMARY")

# Display metrics summary using moe_comprehensive
if 'moe_comprehensive' in globals():
    print(f"{'Metric':<25} {'Value':<15}")
    print(f"{'MMLU Accuracy':<25} {moe_comprehensive['accuracy']:.4f}")
    print(f"{'Top-2 Accuracy':<25} {moe_comprehensive['top2_accuracy']:.4f}")
    print(f"{'ECE (Calibration)':<25} {moe_comprehensive['ece']:.4f}")
    print(f"{'FLOPs (G)':<25} {moe_comprehensive['flops']/1e9:.2f}")
    print(f"{'Tokens/sec':<25} {moe_comprehensive['tokens_per_second']:.2f}")
    print(f"{'ms/token':<25} {moe_comprehensive['ms_per_token']:.2f}")
    print(f"{'Samples/sec':<25} {moe_comprehensive['samples_per_second']:.2f}")
    print(f"{'Total Params (B)':<25} {moe_comprehensive['total_params']/1e9:.2f}")
    print(f"{'Active Params (B)':<25} {moe_comprehensive['active_params']/1e9:.2f}")
    print(f"{'Trainable Params (M)':<25} {moe_comprehensive['trainable_params']/1e6:.2f}")
    print(f"{'Sparsity Ratio':<25} {moe_comprehensive['sparsity_ratio']:.2%}")
    print(f"{'Model Size (MB)':<25} {moe_comprehensive['model_size_mb']:.2f}")
    print(f"{'GPU Allocated (GB)':<25} {moe_comprehensive['gpu_memory_allocated_gb']:.2f}")
    print(f"{'GPU Reserved (GB)':<25} {moe_comprehensive['gpu_memory_reserved_gb']:.2f}")
    print(f"{'Correct/Total':<25} {moe_comprehensive['correct']}/{moe_comprehensive['total']}")

    print("\nMoE Architecture Details:")
    print(f"  Experts per layer: {NUM_EXPERTS}")
    print(f"  Active experts per token: {NUM_EXPERTS_PER_TOK}")
    print(f"  Router aux loss coefficient: {ROUTER_AUX_LOSS_COEF}")
    print(f"  Router jitter noise: {ROUTER_JITTER_NOISE}")
else:
    print("MoE comprehensive results not available. Please run the MoE baseline evaluation first.")


### 15.10 MoE Baseline Cleanup and Finish

Close Weights & Biases run and perform final cleanup for MoE baseline.


## Summary of Enhancements

This notebook now includes:

### 16. Comprehensive Evaluation Metrics 
- **Computational Efficiency**: FLOPs, tokens/sec, ms/token
- **Parameter Efficiency**: Total, active, and trainable parameters
- **Scalability**: Sparsity ratio, memory usage
- Applied to: Baseline, MoE frozen, and MoE trained models

### 2. MoE Variant Experiment System 
- **Configuration Framework**: Easy-to-define experiment configs
- **Predefined Variants**: 4x1, 8x1, 8x2, 16x2, sparse placement
- **Experiment Runner**: Automated variant testing with comprehensive metrics
- **Optional Execution**: Set `RUN_VARIANT_EXPERIMENTS=True` to run

### 3. Experiment Dashboard 
- **Comparison Tables**: Side-by-side metrics for all experiments
- **Visualizations**: Accuracy vs throughput, FLOPs, memory
- **Automated Reports**: Markdown and CSV exports
- **Best Model Identification**: Highlights top performers

### Files Generated:
- `results/baseline_comprehensive.json` - Full baseline metrics
- `results/moe_baseline_comprehensive.json` - MoE frozen metrics
- `results/moe_trained_comprehensive.json` - MoE trained metrics
- `results/efficiency_comparison.png` - Comparison visualizations
- `results/experiment_report.md` - Comprehensive markdown report
- `results/comprehensive_comparison.csv` - Exportable comparison table
- `experiments/{variant}_results.json` - Individual variant results (if run)

### Research Goals Addressed:
 **Efficiency Metrics**: Throughput, FLOPs, latency (ms/token) 
 **Scalability Metrics**: Parameter growth, sparsity, memory 
 **Variant Testing**: Different expert counts, routing, placement 
 **Systematic Comparison**: Automated tracking and visualization 

## 17. Cleanup and Finish

Close Weights & Biases run and perform final cleanup.

## 16.5 Knowledge Distillation (KD) Baseline Comparison


Compare three training configurations on 10% dataset with 250 steps:
1. **No-KD Baseline**: Pure supervised learning without knowledge distillation
2. **KD-Standard**: Output distillation only (kd_alpha=0.5, T=4.0)
3. **KD-Router-Stable**: Output distillation + light routing KD (kd_alpha=0.6, T=5.0, routing_weight=0.1)


Test if KD improves MoE training stabilization and helps identify better configurations.


In [ ]:
# Finish wandb run
try:
    if 'wandb' in globals() and wandb.run is not None:
        wandb.finish()
except Exception as e:
    print(f"WandB finish failed: {e}")

# Close TensorBoard writer
try:
    if 'tb_writer' in globals() and tb_writer is not None:
        tb_writer.close()
except Exception as e:
    print(f"TensorBoard writer close failed: {e}")

# Final memory cleanup
torch.cuda.empty_cache()
gc.collect()

print("\n   All tasks completed successfully!")
print("\n    Output Files:")
print(f"   - Baseline metrics: baseline_metrics.csv")
print(f"   - Baseline config: baseline_config.json")
print(f"   - Visualizations: confusion_matrix.png")
print(f"   - Report: baseline_report.md")
print("\n   Experiment Tracking:")
print("   - Weights & Biases dashboard")

# Ensure log_dir is defined for the print statement, or handle its absence
if 'log_dir' in globals():
    print(f"   - TensorBoard logs: {log_dir}")
else:
    print(f"   - TensorBoard logs: Not available (tb_writer was not initialized)")


## 18. Comprehensive Experiment Comparison Dashboard

Systematic comparison of all experiments:
- Dense baseline vs MoE variants
- Frozen vs trained experts
- Different configurations
- Performance vs efficiency trade-offs

## 16.5 Knowledge Distillation (KD) Baseline Comparison

Compare three training configurations on 10% dataset with 250 steps:
1. **No-KD Baseline**: Pure supervised learning without knowledge distillation
2. **KD-Standard**: Output distillation only (kd_alpha=0.5, T=4.0)
3. **KD-Router-Stable**: Output distillation + light routing KD (kd_alpha=0.6, T=5.0, routing_weight=0.1)

Test if KD improves MoE training stabilization and helps identify better configurations.
